# CS781 Phase 3 — Experiment 6 (v5 STABLE)
## Knowledge Distillation: Model4 + rsLoRA Ensemble Teacher → Phase-1 Model2 Student

**Goal:** distill the strongest available Model4+rsLoRA teacher signal into Phase-1 Model2 while keeping model selection methodologically clean.

### Important distinction
Your best rsLoRA leaderboard result came from the standard **heldout + refit ensemble**. This notebook preserves that information cleanly: heldout-only teacher targets for unbiased DEV selection, then heldout+refit teacher fusion for the final train+dev refit.

You already have:
- `teacher_logits_train_heldout.npz` — heldout rsLoRA teacher logits on TRAIN
- `teacher_logits_train_dev_refit.npz` — refit rsLoRA teacher logits on TRAIN+DEV

This v2 uses them in the strongest **leakage-safe** way:

1. **Heldout KD development**
   - student trains on TRAIN only
   - teacher = heldout rsLoRA teacher only
   - DEV remains genuinely unseen for model selection

2. **Final train+dev KD refit**
   - on TRAIN rows: teacher target = probability ensemble of heldout + refit teachers
   - on DEV rows: teacher target = refit teacher, unless a heldout-teacher DEV-logit file is attached
   - if `teacher_logits_dev_heldout.npz` is available, the final refit uses the exact heldout+refit ensemble teacher on DEV too

This avoids contaminating the heldout DEV selection with a refit teacher that has already seen DEV.

### Student
Exact Phase-1 **Model2**:
- 13-layer decoder-only causal Transformer
- `d_model=640`, 10 heads, FFN 2560
- full student fine-tuning
- layerwise LR decay
- packed BiGRU-256 RR task head

### KD objective
`Loss = 0.50 * HardCE + 0.50 * T² * KL(teacher || student)`

with:
- `T = 2.5`
- class-balanced hard CE
- class-balanced KD loss
- label smoothing on the hard term

### Strong selection / submission strategy
- 4 dev evaluations per epoch
- exact combined 13-label Macro-F1
- top-3 KD student snapshots retained
- exact supplied Phase-3 evaluator before refit
- heldout / refit / ensemble / snapshot ensemble / snapshot+refit ensemble submissions
- strict private-test structural checks

> Required Kaggle files: `dec_full_best*.pt`, exact `tokenizer.json`, `teacher_logits_train_heldout.npz`, `teacher_logits_train_dev_refit.npz`.  
> Optional: `teacher_logits_dev_heldout.npz` for a fully symmetric ensemble teacher during final refit.


### v3 tweaks (for the best score within today's deadline)
- **Teacher emphasis:** `T=3.0`, hard 0.40 / KD 0.60 (the teacher is much stronger than a Model-2 student; softer targets carry more class-similarity information for confusable roles).
- **Speed:** 3 epochs, patience 5, micro-batch 4 × accumulation 4 (same effective batch 16; set 2 × 8 if the T4 runs out of memory).
- **Safety assert** that the concatenated TRAIN+DEV teacher targets are in exactly the same row order and gold labels as `all_labeled_records`.
- **Cell 21b (optional):** `kd_teacher_blend` submission = student log-probs blended with the rsLoRA teacher's test log-probs. It is an extra leaderboard entry, not the KD result to report.


### v4 robustness upgrades
- keeps v3's stronger KD settings: `T=3.0`, hard/KD weights `0.40/0.60`, 3 epochs, effective batch 16;
- keeps the extra TRAIN+DEV row-order assertions;
- the optional student+teacher leaderboard blend now reproduces the rsLoRA teacher's **actual winning raw-logit ensemble** (`0.5*heldout + 0.5*refit`) rather than silently changing the teacher ensemble definition;
- external teacher test logits are aligned by the full `(normalized_doc_id, sent_idx, domain)` key;
- duplicate keys, missing/extra rows, wrong class dimension, NaN/Inf, and ambiguous multiple teacher files are guarded against;
- generates several clearly labeled bonus blend weights while keeping all pure-KD submissions untouched.


### v5 stability fixes
- T4-safe microbatching: `microbatch=2`, `grad_accum=8` (effective batch remains 16).
- lower AMP initial loss scale (`4096`) to reduce early FP16 overflows.
- occasional non-finite FP16 gradient norms are handled by `GradScaler` instead of crashing the run.
- scheduler/global-update counters advance only after a real optimizer update.
- overflow counts are logged and stored in checkpoints/history.
- the final train+dev refit uses the same overflow-safe logic.
- teacher `.npz` discovery accepts Kaggle filenames such as `teacher_logits_train_heldout (1).npz`.


In [17]:
# ============================================================
# CELL 1 — Imports, seed, paths, and KD configuration
# ============================================================
import os, re, gc, json, math, time, random, zipfile, subprocess, sys, hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from huggingface_hub import HfApi, hf_hub_download, login

SEED = 1234
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("Could not read Kaggle HF_TOKEN secret:", repr(e))
assert HF_TOKEN, "HF_TOKEN not found in Kaggle Secrets."
login(token=HF_TOKEN)
HF_REPO = "Exploration-Lab/CS-781-Capstone"

ROOT = Path("/kaggle/working/phase3_exp6_kd_rslora_teacher_model2_student")
ROOT.mkdir(parents=True, exist_ok=True)

BEST_CKPT = ROOT / "best_kd_student.pt"
FINAL_CKPT = ROOT / "final_refit_kd_student.pt"
DEV_PRED = ROOT / "prediction_dev.json"
CL_DEV_TRUTH = ROOT / "cl_dev_truth.json"
IT_DEV_TRUTH = ROOT / "it_dev_truth.json"
LABELS_JSON = ROOT / "labels.json"
EVAL_SCRIPT = ROOT / "evaluate_phase3.py"
OFFICIAL_PRED = ROOT / "prediction.json"
SUBMISSION_ZIP = ROOT / "submission.zip"

# -------- Exact Phase-1 Model 2 --------
VOCAB_SIZE = 24_576
D_MODEL = 640
N_LAYERS = 13
N_HEADS = 10
FFN_DIM = 2560
MAX_POSITIONS = 2048
DROPOUT = 0.0
EXPECTED_TOKENIZER_SHA256 = "9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d"

# -------- Phase-3 context: match the strong teacher setup --------
MAX_LEN = 640
CONTEXT_SENTS = 2
USE_POS = True
TARGET_BUDGET = 320
PREV_BUDGET = 120
NEXT_BUDGET = 120

# -------- Knowledge Distillation / teacher fusion --------
KD_TEMPERATURE = 3.0
HARD_LOSS_WEIGHT = 0.40
KD_LOSS_WEIGHT = 0.60
KD_WEIGHT_BY_CLASS = True
TEACHER_ENSEMBLE_WEIGHT_HELDOUT = 0.50
TEACHER_ENSEMBLE_WEIGHT_REFIT = 0.50
TEACHER_FUSION_SPACE = "probability"  # average probabilities, then take log
assert abs(HARD_LOSS_WEIGHT + KD_LOSS_WEIGHT - 1.0) < 1e-8

# -------- Student optimization --------
STUDENT_BACKBONE_LR = 4e-5
STUDENT_LAYERWISE_LR_DECAY = 0.92
STUDENT_HEAD_LR = 7e-4
HEAD_HIDDEN = 256
HEAD_DROPOUT = 0.10

NUM_EPOCHS = 3
EVAL_PER_EPOCH = 4
EARLY_STOP_PATIENCE_EVALS = 5

# Full 80M student is trainable; keep T4-safe effective batch 16.
MICRO_BATCH_SIZE = 2      # if CUDA OOM on T4: set 2 and GRAD_ACCUM_STEPS = 8
EVAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 8

WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
GRAD_CLIP = 1.0
USE_AMP = True
AMP_INIT_SCALE = 4096.0
AMP_GROWTH_FACTOR = 2.0
AMP_BACKOFF_FACTOR = 0.5
AMP_GROWTH_INTERVAL = 2000
LABEL_SMOOTHING = 0.02

CLASS_WEIGHT_MIN = 0.35
CLASS_WEIGHT_MAX = 3.50

PRINT_EVERY = 250
NUM_WORKERS = 2
LENGTH_GROUPED_TRAINING = True
LENGTH_BUCKET_MULT = 64
NAN_CHECK_EVERY = 50

REFIT_MATCH_ORIGINAL_SCHEDULE = True
ENSEMBLE_HELDOUT_AND_REFIT = True
TOPK_HELDOUT_SNAPSHOTS = 3
MAKE_SNAPSHOT_ENSEMBLES = True

# Optional leaderboard-only student+teacher blends.
# These are NOT the official pure-KD result.
BONUS_TEACHER_BLEND_STUDENT_WEIGHTS = [0.30, 0.40, 0.50]

print("Output directory:", ROOT)
print(f"Student Model2: {N_LAYERS} layers | d={D_MODEL} | heads={N_HEADS} | FFN={FFN_DIM}")
print(f"KD: T={KD_TEMPERATURE} | hard={HARD_LOSS_WEIGHT} | soft={KD_LOSS_WEIGHT}")
print(f"Student top-block LR={STUDENT_BACKBONE_LR} | decay={STUDENT_LAYERWISE_LR_DECAY} | head LR={STUDENT_HEAD_LR}")
print("Context ±", CONTEXT_SENTS, "| MAX_LEN:", MAX_LEN)


Device: cuda
GPU: Tesla T4
Output directory: /kaggle/working/phase3_exp6_kd_rslora_teacher_model2_student
Student Model2: 13 layers | d=640 | heads=10 | FFN=2560
KD: T=3.0 | hard=0.4 | soft=0.6
Student top-block LR=4e-05 | decay=0.92 | head LR=0.0007
Context ± 2 | MAX_LEN: 640


In [18]:
# ============================================================
# CELL 2 — Discover Phase-1 Model2 + teacher-logit inputs
# ============================================================
def find_files(predicate):
    out=[]
    for base in ["/kaggle/input", "/kaggle/working"]:
        if not os.path.isdir(base): continue
        for root, _, files in os.walk(base):
            for fn in files:
                p=os.path.join(root,fn)
                if predicate(fn,p):
                    out.append(p)
    return out

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(1<<20),b""):
            h.update(b)
    return h.hexdigest()

student_ckpts=find_files(
    lambda fn,p: fn.lower().endswith(".pt") and "dec_full_best" in fn.lower()
)
tok_matches=find_files(
    lambda fn,p: fn.lower().endswith(".json") and "tokenizer" in fn.lower()
)
def npz_name_matches(fn, stem):
    low=fn.lower()
    return (
        low==f"{stem}.npz"
        or (
            low.startswith(stem.lower()+" ")
            and low.endswith(".npz")
        )
    )

teacher_train_matches=find_files(
    lambda fn,p: npz_name_matches(fn,"teacher_logits_train_heldout")
)
teacher_refit_matches=find_files(
    lambda fn,p: npz_name_matches(fn,"teacher_logits_train_dev_refit")
)
teacher_dev_matches=find_files(
    lambda fn,p: npz_name_matches(fn,"teacher_logits_dev_heldout")
)

print("Student checkpoint candidates:")
for p in student_ckpts: print(" ",p)
print("Teacher TRAIN logits candidates:")
for p in teacher_train_matches: print(" ",p)
print("Teacher TRAIN+DEV logits candidates:")
for p in teacher_refit_matches: print(" ",p)
print("Optional heldout-teacher DEV logits candidates:")
for p in teacher_dev_matches: print(" ",p)

assert student_ckpts, "Attach Phase-1 Model2 dec_full_best*.pt"
assert tok_matches, "Attach exact tokenizer.json"
assert teacher_train_matches, "Attach teacher_logits_train_heldout.npz from rsLoRA run"
assert teacher_refit_matches, "Attach teacher_logits_train_dev_refit.npz from rsLoRA run"

def score_path(p,words):
    low=p.lower()
    return sum(w in low for w in words)

PHASE1_CKPT=sorted(
    student_ckpts,
    key=lambda p:(score_path(p,["model2","phase1","decoder","dec_full_best"]),p),
    reverse=True
)[0]

exact_tok=[p for p in tok_matches if sha256_file(p)==EXPECTED_TOKENIZER_SHA256]
TOKENIZER_FILE=exact_tok[0] if exact_tok else sorted(tok_matches)[0]

# Prefer rsLoRA-named paths if multiple runs are attached.
TEACHER_TRAIN_PATH=sorted(
    teacher_train_matches,
    key=lambda p:(score_path(p,["rslora","model4","teacher"]),p),
    reverse=True
)[0]
TEACHER_REFIT_PATH=sorted(
    teacher_refit_matches,
    key=lambda p:(score_path(p,["rslora","model4","teacher"]),p),
    reverse=True
)[0]
TEACHER_DEV_HELDOUT_PATH=(
    sorted(
        teacher_dev_matches,
        key=lambda p:(score_path(p,["rslora","model4","teacher"]),p),
        reverse=True
    )[0]
    if teacher_dev_matches else None
)

assert sha256_file(TOKENIZER_FILE)==EXPECTED_TOKENIZER_SHA256

print("\nUSING")
print(" Student Phase1:",PHASE1_CKPT)
print(" Tokenizer:",TOKENIZER_FILE)
print(" Teacher train:",TEACHER_TRAIN_PATH)
print(" Teacher train+dev:",TEACHER_REFIT_PATH)
print(" Teacher heldout DEV (optional):",TEACHER_DEV_HELDOUT_PATH)
print(" Tokenizer SHA:",sha256_file(TOKENIZER_FILE))


Student checkpoint candidates:
  /kaggle/input/datasets/aaryant23/phase3-kd-model4-2/dec_full_best (8).pt
Teacher TRAIN logits candidates:
  /kaggle/input/datasets/aaryant23/phase3-kd-model4-2/teacher_logits_train_heldout.npz
Teacher TRAIN+DEV logits candidates:
  /kaggle/input/datasets/aaryant23/phase3-kd-model4-2/teacher_logits_train_dev_refit.npz
Optional heldout-teacher DEV logits candidates:

USING
 Student Phase1: /kaggle/input/datasets/aaryant23/phase3-kd-model4-2/dec_full_best (8).pt
 Tokenizer: /kaggle/input/datasets/aaryant23/phase3-kd-model4-2/tokenizer.json
 Teacher train: /kaggle/input/datasets/aaryant23/phase3-kd-model4-2/teacher_logits_train_heldout.npz
 Teacher train+dev: /kaggle/input/datasets/aaryant23/phase3-kd-model4-2/teacher_logits_train_dev_refit.npz
 Teacher heldout DEV (optional): None
 Tokenizer SHA: 9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d


In [19]:
# ============================================================
# CELL 3 — Original custom byte-level LegalBPE
# ============================================================

BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class LegalBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = int(vocab_size)
        self.merges = []
        self.itos = list(BPE_SPECIALS)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            self.symbol_bytes[s] = bytes([b])
            self.stoi[s] = len(self.itos)
            self.itos.append(s)
        self._reset_encode_cache()

    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)

    @staticmethod
    def word_to_symbols(word):
        return [f"{b:02x}" for b in word.encode("utf-8")]

    def _reset_encode_cache(self):
        self._ranks = {(l,r): i for i,(l,r) in enumerate(self.merges)}
        self._word_cache = {}
        self._cache_nmerges = len(self.merges)

    def encode_word(self, word):
        if self._cache_nmerges != len(self.merges):
            self._reset_encode_cache()

        hit = self._word_cache.get(word)
        if hit is not None:
            return list(hit)

        symbols = self.word_to_symbols(word)
        ranks = self._ranks

        while len(symbols) > 1:
            best = None
            best_rank = None
            for j in range(len(symbols)-1):
                r = ranks.get((symbols[j], symbols[j+1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best = (symbols[j], symbols[j+1])
                    best_rank = r
            if best is None:
                break

            left, right = best
            merged = []
            j = 0
            while j < len(symbols):
                if j+1 < len(symbols) and symbols[j] == left and symbols[j+1] == right:
                    merged.append(left+right)
                    j += 2
                else:
                    merged.append(symbols[j])
                    j += 1
            symbols = merged

        if len(self._word_cache) >= 250_000:
            self._word_cache.clear()
        self._word_cache[word] = tuple(symbols)
        return symbols

    def encode(self, text):
        out = []
        unk = self.unk_token_id
        for unit in re.findall(r"\S+|\s+", text):
            for s in self.encode_word(unit):
                out.append(self.stoi.get(s, unk))
        return out

    def decode(self, ids):
        buf = bytearray()
        for idx in ids:
            idx = int(idx)
            tok = self.itos[idx]
            if tok in BPE_SPECIALS:
                continue
            buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8", errors="replace")

    @classmethod
    def load_json(cls, path):
        obj = json.load(open(path, encoding="utf-8"))
        tok = cls(obj["vocab_size"])
        tok.merges = [list(x) for x in obj["merges"]]
        tok.itos = list(obj["itos"])
        tok.stoi = {t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size = len(tok.itos)

        tok.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            tok.symbol_bytes[s] = bytes([b])
        for l, r in tok.merges:
            tok.symbol_bytes[l+r] = tok.symbol_bytes[l] + tok.symbol_bytes[r]

        tok._reset_encode_cache()
        return tok

tokenizer = LegalBPE.load_json(TOKENIZER_FILE)
print("Tokenizer vocab:", len(tokenizer))
assert len(tokenizer) == VOCAB_SIZE, (len(tokenizer), VOCAB_SIZE)

probe = "IN THE SUPREME COURT OF INDIA\nSection 14: legal test."
assert tokenizer.decode(tokenizer.encode(probe)) == probe
print("✅ tokenizer round-trip passed")


Tokenizer vocab: 24576
✅ tokenizer round-trip passed


In [20]:
# ============================================================
# CELL 4 — Exact Phase-1 Model 2 decoder-only architecture
# ============================================================
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_positions):
        super().__init__()
        pos = torch.arange(max_positions, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_positions, d_model, dtype=torch.float32)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, position_ids):
        return self.pe[position_ids]


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, past_kv=None, use_cache=False, key_valid=None):
        B, T, C = x.shape

        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        past_len = 0
        if past_kv is not None:
            pk, pv = past_kv
            past_len = pk.size(2)
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)

        attn_mask = None
        is_causal = False

        if key_valid is not None:
            key_mask = key_valid[:, None, None, :].bool()
            if past_len == 0 and T > 1:
                causal = torch.ones((T, T), dtype=torch.bool, device=x.device).tril()
                attn_mask = key_mask & causal[None, None, :, :]
                # Protect fully-masked padded query rows without changing real-token attention.
                eye = torch.eye(T, dtype=torch.bool, device=x.device)[None, None, :, :]
                attn_mask = attn_mask | eye
            else:
                attn_mask = key_mask
        elif past_kv is None:
            is_causal = True

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal,
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.out_proj(y)

        new_kv = (k, v) if use_cache else None
        return y, new_kv


class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout),
        )
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, past_kv=None, use_cache=False, key_valid=None):
        a, new_kv = self.attn(
            self.ln1(x),
            past_kv=past_kv,
            use_cache=use_cache,
            key_valid=key_valid,
        )
        x = x + self.resid_drop(a)
        x = x + self.ff(self.ln2(x))
        return x, new_kv


class DecoderOnlyLM(nn.Module):
    def __init__(
        self, vocab_size, d_model, n_layers, n_heads,
        ffn_dim, dropout, max_positions
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_positions = max_positions

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos = SinusoidalPositionalEncoding(d_model, max_positions)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            DecoderBlock(d_model, n_heads, ffn_dim, dropout)
            for _ in range(n_layers)
        ])
        self.final_ln = nn.LayerNorm(d_model)

        self.apply(self._init_weights)

        scaled_std = 0.02 / math.sqrt(2 * n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.out_proj.weight, mean=0.0, std=scaled_std)
            nn.init.normal_(block.ff[2].weight, mean=0.0, std=scaled_std)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward_hidden(
        self, input_ids, past_key_values=None, use_cache=False,
        position_ids=None, key_valid=None
    ):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)

        if past_len + T > self.max_positions:
            raise ValueError(
                f"context {past_len + T} exceeds max_positions={self.max_positions}"
            )

        if position_ids is None:
            position_ids = torch.arange(
                past_len, past_len + T, device=input_ids.device
            )[None, :].expand(B, T)

        x = self.token_emb(input_ids) + self.pos(position_ids).to(self.token_emb.weight.dtype)
        x = self.drop(x)

        new_cache = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            x, kv = block(
                x,
                past_kv=past,
                use_cache=use_cache,
                key_valid=key_valid,
            )
            if use_cache:
                new_cache.append(kv)

        x = self.final_ln(x)
        return x, new_cache

    def project(self, hidden):
        return F.linear(hidden, self.token_emb.weight)


def build_backbone():
    return DecoderOnlyLM(
        VOCAB_SIZE, D_MODEL, N_LAYERS, N_HEADS,
        FFN_DIM, DROPOUT, MAX_POSITIONS
    )

backbone = build_backbone()
n_backbone = sum(p.numel() for p in backbone.parameters())
print("Backbone params:", f"{n_backbone/1e6:.3f}M")
assert 75_000_000 <= n_backbone <= 85_000_000, n_backbone


Backbone params: 79.736M


In [21]:
# ============================================================
# CELL 5 — Load Phase-1 Model 2 checkpoint strictly
# ============================================================
obj = torch.load(PHASE1_CKPT, map_location="cpu", weights_only=False)
print("Checkpoint keys:", list(obj.keys()) if isinstance(obj, dict) else type(obj))
state = obj["model"] if isinstance(obj, dict) and "model" in obj else obj

missing, unexpected = backbone.load_state_dict(state, strict=True)
assert not missing and not unexpected
print("✅ strict=True checkpoint load succeeded")

if isinstance(obj, dict):
    print(
        "step:", obj.get("global_step", obj.get("step")),
        "val_loss:", obj.get("val_loss", obj.get("best_val"))
    )
    cfg = obj.get("config", {})
    if cfg:
        print("Checkpoint architecture:", cfg.get("architecture"))
        for k, expected in [
            ("d_model", D_MODEL),
            ("n_layers", N_LAYERS),
            ("n_heads", N_HEADS),
            ("ffn_dim", FFN_DIM),
            ("max_positions", MAX_POSITIONS),
        ]:
            if k in cfg:
                assert int(cfg[k]) == int(expected), (k, cfg[k], expected)

backbone = backbone.to(DEVICE)
# KD fine-tunes the full smaller student, with layerwise LR decay.
for p in backbone.parameters():
    p.requires_grad = True
print("Student Phase-1 backbone trainable params:", sum(p.numel() for p in backbone.parameters() if p.requires_grad))



Checkpoint keys: ['model', 'global_step', 'val_loss', 'config']
✅ strict=True checkpoint load succeeded
step: 16266 val_loss: 1.7356935616699047
Checkpoint architecture: decoder_only_causal
Student Phase-1 backbone trainable params: 79735680


In [22]:
# ============================================================
# CELL 6 — Teacher-logit alignment and KD safety checks
# ============================================================
def teacher_key(doc_id,sent_idx,domain):
    return (normalize_id(str(doc_id)),int(sent_idx),str(domain))

def load_teacher_npz(path):
    z=np.load(path,allow_pickle=True)
    required={"logits","doc_id","sent_idx","domain"}
    assert required.issubset(set(z.files)),(path,z.files)

    logits=np.asarray(z["logits"],dtype=np.float32)
    assert logits.ndim==2 and logits.shape[1]==NUM_LABELS if "NUM_LABELS" in globals() else logits.ndim==2

    keys=[
        teacher_key(d,s,dom)
        for d,s,dom in zip(z["doc_id"],z["sent_idx"],z["domain"])
    ]
    assert len(set(keys))==len(keys),f"Duplicate teacher keys in {path}"

    y=np.asarray(z["y"],dtype=np.int64) if "y" in z.files else None
    if y is not None:
        assert len(y)==len(logits)

    return {
        "path":str(path),
        "logits":logits,
        "keys":keys,
        "y":y,
    }

def align_teacher_to_records(teacher_obj,records,require_gold_match=True):
    src={k:i for i,k in enumerate(teacher_obj["keys"])}
    order=[]
    for r in records:
        k=teacher_key(r["doc_id"],r["sent_idx"],r["domain"])
        assert k in src,f"Teacher missing record {k}"
        order.append(src[k])

    order=np.asarray(order,dtype=np.int64)
    logits=teacher_obj["logits"][order]
    assert logits.shape==(len(records),NUM_LABELS)

    if require_gold_match and teacher_obj["y"] is not None:
        ty=teacher_obj["y"][order]
        gy=np.asarray([r["label_id"] for r in records],dtype=np.int64)
        assert np.array_equal(ty,gy),(
            "Teacher y does not align with current label IDs. "
            "Check tokenizer/data/label-vocab provenance."
        )

    assert np.isfinite(logits).all(),"Teacher logits contain NaN/Inf"
    return logits

print("Teacher alignment helpers ready.")


def teacher_probs(logits):
    x=np.asarray(logits,dtype=np.float64)
    x=x-x.max(axis=-1,keepdims=True)
    e=np.exp(x)
    return e/e.sum(axis=-1,keepdims=True)

def fuse_teacher_logits(logits_a,logits_b,w_a=0.5,w_b=0.5):
    """
    Probability-space teacher ensemble.
    Returns log-probabilities, which are valid teacher logits up to an additive
    constant for softmax/KL distillation.
    """
    assert logits_a.shape==logits_b.shape
    assert abs((w_a+w_b)-1.0)<1e-8
    pa=teacher_probs(logits_a)
    pb=teacher_probs(logits_b)
    p=w_a*pa+w_b*pb
    p=np.clip(p,1e-9,1.0)
    return np.log(p).astype(np.float32)

def build_final_refit_teacher_targets(train_records,dev_records):
    """
    TRAIN:
      exact heldout+refit rsLoRA ensemble teacher.

    DEV:
      exact ensemble if heldout DEV logits are attached;
      otherwise refit teacher only (safe because DEV is no longer used for selection).
    """
    held_obj=load_teacher_npz(TEACHER_TRAIN_PATH)
    refit_obj=load_teacher_npz(TEACHER_REFIT_PATH)

    held_train=align_teacher_to_records(
        held_obj,train_records,require_gold_match=True
    )
    refit_train=align_teacher_to_records(
        refit_obj,train_records,require_gold_match=True
    )

    train_fused=fuse_teacher_logits(
        held_train,refit_train,
        TEACHER_ENSEMBLE_WEIGHT_HELDOUT,
        TEACHER_ENSEMBLE_WEIGHT_REFIT,
    )

    if TEACHER_DEV_HELDOUT_PATH is not None:
        held_dev_obj=load_teacher_npz(TEACHER_DEV_HELDOUT_PATH)
        held_dev=align_teacher_to_records(
            held_dev_obj,dev_records,require_gold_match=True
        )
        refit_dev=align_teacher_to_records(
            refit_obj,dev_records,require_gold_match=True
        )
        dev_targets=fuse_teacher_logits(
            held_dev,refit_dev,
            TEACHER_ENSEMBLE_WEIGHT_HELDOUT,
            TEACHER_ENSEMBLE_WEIGHT_REFIT,
        )
        dev_mode="heldout+refit probability ensemble"
    else:
        dev_targets=align_teacher_to_records(
            refit_obj,dev_records,require_gold_match=True
        )
        dev_mode="refit teacher only (heldout DEV logits not attached)"

    final=np.concatenate(
        [train_fused,dev_targets],
        axis=0,
    ).astype(np.float32)

    print("Final-refit teacher construction:")
    print(" TRAIN:", "heldout+refit probability ensemble")
    print(" DEV  :", dev_mode)
    print(" shape:", final.shape)
    return final,dev_mode

print("Teacher ensemble helpers ready.")


Teacher alignment helpers ready.
Teacher ensemble helpers ready.


In [23]:
# ============================================================
# CELL 7 — Download Phase-3 raw JSON files (bypass datasets CastError)
# ============================================================

api = HfApi()
repo_files = api.list_repo_files(
    repo_id=HF_REPO,
    repo_type="dataset",
    token=HF_TOKEN,
)

def exact_repo_file(filename):
    matches = [
        f for f in repo_files
        if os.path.basename(f).lower() == filename.lower()
    ]
    assert len(matches) == 1, (filename, matches)
    return matches[0]

FILES = {
    "CL_train": exact_repo_file("CL_train.json"),
    "CL_dev": exact_repo_file("CL_dev.json"),
    "IT_train": exact_repo_file("IT_train.json"),
    "IT_dev": exact_repo_file("IT_dev.json"),
    "CL_test": exact_repo_file("CL_test.json"),
    "IT_test": exact_repo_file("IT_test.json"),
    "label_vocab": exact_repo_file("label_vocab.json"),
}

print(json.dumps(FILES, indent=2))

def load_repo_json(repo_path):
    p = hf_hub_download(
        repo_id=HF_REPO,
        filename=repo_path,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

cl_train_raw = load_repo_json(FILES["CL_train"])
cl_dev_raw   = load_repo_json(FILES["CL_dev"])
it_train_raw = load_repo_json(FILES["IT_train"])
it_dev_raw   = load_repo_json(FILES["IT_dev"])
cl_test_raw  = load_repo_json(FILES["CL_test"])
it_test_raw  = load_repo_json(FILES["IT_test"])
label_vocab_raw = load_repo_json(FILES["label_vocab"])

assert isinstance(cl_train_raw, list) and isinstance(it_train_raw, list)
assert isinstance(cl_dev_raw, list) and isinstance(it_dev_raw, list)
assert isinstance(cl_test_raw, dict) and isinstance(it_test_raw, dict)

print("CL train/dev:", len(cl_train_raw), len(cl_dev_raw))
print("IT train/dev:", len(it_train_raw), len(it_dev_raw))
print("CL/IT test docs:", len(cl_test_raw), len(it_test_raw))


{
  "CL_train": "Phase_3_RR/Train/CL_train.json",
  "CL_dev": "Phase_3_RR/Train/CL_dev.json",
  "IT_train": "Phase_3_RR/Train/IT_train.json",
  "IT_dev": "Phase_3_RR/Train/IT_dev.json",
  "CL_test": "Phase_3_RR/Test/CL_test.json",
  "IT_test": "Phase_3_RR/Test/IT_test.json",
  "label_vocab": "Phase_3_RR/Train/label_vocab.json"
}
CL train/dev: 40 5
IT train/dev: 40 5
CL/IT test docs: 5 5


In [24]:
# ============================================================
# CELL 8 — Labels, examples, class weights, and data checks
# ============================================================

def parse_label_vocab(data):
    if isinstance(data, list):
        labels = list(data)
        label_to_id = {x:i for i,x in enumerate(labels)}
    elif isinstance(data, dict):
        vals = list(data.values())
        if all(isinstance(v, int) and not isinstance(v, bool) for v in vals):
            label_to_id = {str(k): int(v) for k, v in data.items()}
            assert sorted(label_to_id.values()) == list(range(len(label_to_id)))
            labels = [None] * len(label_to_id)
            for lab, idx in label_to_id.items():
                labels[idx] = lab
        elif all(isinstance(v, str) for v in vals):
            tmp = {int(k): v for k, v in data.items()}
            labels = [tmp[i] for i in sorted(tmp)]
            label_to_id = {x:i for i,x in enumerate(labels)}
        else:
            raise ValueError("Unsupported label_vocab format")
    else:
        raise ValueError("Unsupported label_vocab format")

    return labels, label_to_id

LABELS, LABEL_TO_ID = parse_label_vocab(label_vocab_raw)
ID_TO_LABEL = {v:k for k,v in LABEL_TO_ID.items()}
NUM_LABELS = len(LABELS)

assert NUM_LABELS == 13
print("Labels:")
for i, x in enumerate(LABELS):
    print(i, x)

def normalize_id(doc_id):
    doc_id = str(doc_id)
    return doc_id[:-4] if doc_id.endswith(".txt") else doc_id

def validate_labeled_docs(docs, name):
    ids = set()
    n_sent = 0
    for d in docs:
        did = normalize_id(d["id"])
        assert did not in ids, f"{name}: duplicate ID {did}"
        ids.add(did)
        assert isinstance(d["text"], list)
        assert isinstance(d["labels"], list)
        assert len(d["text"]) == len(d["labels"]), did
        assert all(x in LABEL_TO_ID for x in d["labels"]), did
        n_sent += len(d["text"])
    print(f"{name}: docs={len(docs)} sentences={n_sent}")
    return ids

cltr_ids = validate_labeled_docs(cl_train_raw, "CL_train")
cldv_ids = validate_labeled_docs(cl_dev_raw, "CL_dev")
ittr_ids = validate_labeled_docs(it_train_raw, "IT_train")
itdv_ids = validate_labeled_docs(it_dev_raw, "IT_dev")

assert not (cltr_ids & cldv_ids)
assert not (ittr_ids & itdv_ids)
# No document may silently appear in both legal domains/splits after `.txt` normalization.
assert not (cltr_ids & ittr_ids), "CL_train / IT_train ID overlap"
assert not (cltr_ids & itdv_ids), "CL_train / IT_dev ID overlap"
assert not (cldv_ids & ittr_ids), "CL_dev / IT_train ID overlap"
assert not (cldv_ids & itdv_ids), "CL_dev / IT_dev ID overlap"
assert not ((cltr_ids | cldv_ids) & (ittr_ids | itdv_ids)), "Cross-domain labeled ID overlap"

def position_features(i, n):
    if not USE_POS:
        return [0.0, 0.0, 0.0, 0.0]
    return [i / max(1, n - 1), math.log1p(n) / 8.0, float(i == 0), float(i == n - 1)]

# Flatten into sentence-level records while preserving document context.
def make_labeled_records(docs, domain):
    out = []
    for d in docs:
        sents = d["text"]
        labs = d["labels"]
        did = normalize_id(d["id"])
        for i, (sent, lab) in enumerate(zip(sents, labs)):
            out.append({
                "doc_id": did,
                "sent_idx": i,
                "domain": domain,
                "prev": " ".join(sents[max(0, i-CONTEXT_SENTS):i]) if CONTEXT_SENTS > 0 else "",
                "target": sent,
                "next": " ".join(sents[i+1:i+1+CONTEXT_SENTS]) if CONTEXT_SENTS > 0 else "",
                "pos": position_features(i, len(sents)),
                "label": lab,
                "label_id": LABEL_TO_ID[lab],
            })
    return out

train_records = make_labeled_records(cl_train_raw, "CL") + make_labeled_records(it_train_raw, "IT")
dev_records   = make_labeled_records(cl_dev_raw, "CL") + make_labeled_records(it_dev_raw, "IT")

print("Train sentence records:", len(train_records))
print("Dev sentence records  :", len(dev_records))

train_counts = Counter(x["label"] for x in train_records)
print("\nTrain distribution:")
for lab in LABELS:
    print(f"{lab:30s} {train_counts[lab]:6d}")

# sqrt inverse-frequency; normalize mean to 1; then cap extremes.
N = len(train_records)
C = NUM_LABELS
raw_w = np.array(
    [math.sqrt(N / (C * max(train_counts[lab], 1))) for lab in LABELS],
    dtype=np.float32
)
raw_w = raw_w / raw_w.mean()
class_w = np.clip(raw_w, CLASS_WEIGHT_MIN, CLASS_WEIGHT_MAX)
class_w = torch.tensor(class_w, dtype=torch.float32, device=DEVICE)

print("\nClass weights:")
for lab, w in zip(LABELS, class_w.detach().cpu().tolist()):
    print(f"{lab:30s} {w:.4f}")


Labels:
0 Fact
1 Issue
2 ArgumentPetitioner
3 ArgumentRespondent
4 Statute
5 Dissent
6 PrecedentReliedUpon
7 PrecedentNotReliedUpon
8 PrecedentOverruled
9 RulingByLowerCourt
10 RatioOfTheDecision
11 RulingByPresentCourt
12 None
CL_train: docs=40 sentences=10593
CL_dev: docs=5 sentences=1396
IT_train: docs=40 sentences=6252
IT_dev: docs=5 sentences=746
Train sentence records: 16845
Dev sentence records  : 2142

Train distribution:
Fact                             4098
Issue                             107
ArgumentPetitioner               1613
ArgumentRespondent               1348
Statute                          1099
Dissent                           284
PrecedentReliedUpon              1945
PrecedentNotReliedUpon            569
PrecedentOverruled                  3
RulingByLowerCourt                874
RatioOfTheDecision               4301
RulingByPresentCourt              307
None                              297

Class weights:
Fact                           0.3500
Issue             

In [25]:
# ============================================================
# CELL 9 — Target-preserving context tokenization
# ============================================================
_ENC_CACHE={}
def enc(text):
    hit=_ENC_CACHE.get(text)
    if hit is None:
        hit=tokenizer.encode(text); _ENC_CACHE[text]=hit
    return hit

def crop_head_tail(ids,budget):
    if len(ids)<=budget: return list(ids)
    if budget<=0: return []
    a=budget//2; return list(ids[:a])+list(ids[-(budget-a):])

def build_context_item(domain,prev_text,target_text,next_text,max_len=MAX_LEN):
    # Markers are ordinary text; no tokenizer-vocabulary changes.
    prefix=enc(f"Domain: {domain}\nPrevious context:\n")
    tmark=enc("\nTarget sentence:\n")
    nmark=enc("\nNext context:\n")
    suffix=enc("\nRhetorical role:\n")
    prev=enc(prev_text)[-PREV_BUDGET:] if prev_text else []
    targ=crop_head_tail(enc(target_text),TARGET_BUDGET)
    nxt=enc(next_text)[:NEXT_BUDGET] if next_text else []

    fixed=len(prefix)+len(tmark)+len(nmark)+len(suffix)
    avail=max_len-fixed
    if avail<16: raise RuntimeError("Marker text unexpectedly consumes MAX_LEN")
    tkeep=min(len(targ), max(16,min(TARGET_BUDGET,avail)))
    remain=avail-tkeep
    pkeep=min(len(prev),remain//2); nkeep=min(len(nxt),remain-pkeep)
    spare=remain-pkeep-nkeep
    add=min(spare,len(prev)-pkeep); pkeep+=add; spare-=add
    add=min(spare,len(nxt)-nkeep); nkeep+=add; spare-=add
    prev=prev[-pkeep:] if pkeep else []; targ=crop_head_tail(targ,tkeep); nxt=nxt[:nkeep] if nkeep else []

    ids=prefix+prev+tmark
    target_start=len(ids)
    ids += targ
    target_end=len(ids)
    ids += nmark+nxt+suffix
    if len(ids)>max_len: raise RuntimeError((len(ids),max_len))
    assert target_end>target_start
    return ids,target_start,target_end

lengths=[]; target_lengths=[]
for r in train_records[:1000]+dev_records[:500]:
    ids,a,b=build_context_item(r['domain'],r['prev'],r['target'],r['next'])
    lengths.append(len(ids)); target_lengths.append(b-a)
print("Context-token sample: mean=",round(float(np.mean(lengths)),2),"p95=",round(float(np.percentile(lengths,95)),2),"max=",max(lengths))
print("Target-token sample : mean=",round(float(np.mean(target_lengths)),2),"p95=",round(float(np.percentile(target_lengths,95)),2))


Context-token sample: mean= 318.94 p95= 405.0 max= 595
Target-token sample : mean= 70.4 p95= 144.0


In [26]:
# ============================================================
# CELL 10 — KD datasets + full Model2 student with packed BiGRU head
# ============================================================
# Leakage-safe model-selection stage: use ONLY heldout teacher logits on TRAIN.
# Do NOT mix refit-teacher logits here because the refit teacher has seen DEV.
_teacher_train_obj=load_teacher_npz(TEACHER_TRAIN_PATH)
teacher_train_logits=align_teacher_to_records(
    _teacher_train_obj,train_records,require_gold_match=True
)
print(
    "Aligned heldout teacher logits:",
    teacher_train_logits.shape,
    "| teacher train argmax acc:",
    round(float((teacher_train_logits.argmax(-1)==np.asarray(
        [r["label_id"] for r in train_records]
    )).mean()),4)
)

class RRDataset(Dataset):
    """Tokenize each sentence/context once; optionally attach teacher logits."""
    def __init__(self,records,labeled=True,teacher_logits=None):
        assert VOCAB_SIZE<=65535
        self.records=records
        self.labeled=labeled
        self.teacher_logits=teacher_logits
        if teacher_logits is not None:
            assert len(teacher_logits)==len(records)

        self.items=[]
        self.lengths=[]
        for r in records:
            ids,a,b=build_context_item(
                r["domain"],r["prev"],r["target"],r["next"]
            )
            self.items.append(
                (np.asarray(ids,dtype=np.uint16),a,b)
            )
            self.lengths.append(len(ids))

    def __len__(self):
        return len(self.records)

    def __getitem__(self,idx):
        r=self.records[idx]
        ids,a,b=self.items[idx]
        item={
            "input_ids":ids,
            "target_start":a,
            "target_end":b,
            "doc_id":r["doc_id"],
            "sent_idx":r["sent_idx"],
            "domain":r["domain"],
            "pos":r["pos"],
        }
        if self.labeled:
            item["label_id"]=r["label_id"]
        if self.teacher_logits is not None:
            item["teacher_logits"]=self.teacher_logits[idx]
        return item


def collate_rr(batch):
    T=max(len(x["input_ids"]) for x in batch)
    B=len(batch)

    ids=torch.full(
        (B,T),tokenizer.pad_token_id,dtype=torch.long
    )
    valid=torch.zeros((B,T),dtype=torch.bool)
    tmask=torch.zeros((B,T),dtype=torch.bool)

    pos=torch.tensor(
        [x["pos"] for x in batch],dtype=torch.float32
    )
    domain_id=torch.tensor(
        [0 if x["domain"]=="CL" else 1 for x in batch],
        dtype=torch.long
    )

    for i,x in enumerate(batch):
        z=torch.from_numpy(x["input_ids"].astype(np.int64))
        L=len(z)
        ids[i,:L]=z
        valid[i,:L]=True
        tmask[i,x["target_start"]:x["target_end"]]=True

    out={
        "input_ids":ids,
        "key_valid":valid,
        "target_mask":tmask,
        "pos":pos,
        "domain_id":domain_id,
        "doc_id":[x["doc_id"] for x in batch],
        "sent_idx":[x["sent_idx"] for x in batch],
        "domain":[x["domain"] for x in batch],
    }

    if "label_id" in batch[0]:
        out["labels"]=torch.tensor(
            [x["label_id"] for x in batch],dtype=torch.long
        )

    if "teacher_logits" in batch[0]:
        out["teacher_logits"]=torch.tensor(
            np.stack([x["teacher_logits"] for x in batch]),
            dtype=torch.float32,
        )

    return out


HEAD_PREFIXES=(
    "sequence_gru.",
    "head_norm.",
    "pos_mlp.",
    "domain_emb.",
    "fc1.",
    "classifier.",
)

class KDStudent(nn.Module):
    """
    Full Phase-1 Model2 student + packed BiGRU sentence head.

    The transformer remains causal exactly as pretrained.
    The downstream BiGRU gives the task head access to both directions of the
    already-computed token representations without modifying transformer masks.
    """
    def __init__(self,backbone,num_labels=13,dropout=HEAD_DROPOUT):
        super().__init__()
        self.backbone=backbone
        self.sequence_gru=nn.GRU(
            input_size=D_MODEL,
            hidden_size=HEAD_HIDDEN,
            batch_first=True,
            bidirectional=True,
        )
        feat_dim=2*HEAD_HIDDEN
        self.head_norm=nn.LayerNorm(2*feat_dim)
        self.pos_mlp=nn.Sequential(
            nn.Linear(4,64),
            nn.GELU(),
        )
        self.domain_emb=nn.Embedding(2,32)
        self.fc1=nn.Linear(2*feat_dim+64+32,384)
        self.drop=nn.Dropout(dropout)
        self.classifier=nn.Linear(384,num_labels)

    def forward(self,input_ids,key_valid,target_mask,pos,domain_id):
        h,_=self.backbone.forward_hidden(
            input_ids,
            key_valid=key_valid,
            use_cache=False,
        )

        lengths=key_valid.sum(1).clamp(min=1).cpu()
        packed=nn.utils.rnn.pack_padded_sequence(
            self.drop(h),
            lengths,
            batch_first=True,
            enforce_sorted=False,
        )
        packed_out,_=self.sequence_gru(packed)
        h2,_=nn.utils.rnn.pad_packed_sequence(
            packed_out,
            batch_first=True,
            total_length=h.size(1),
        )
        h2=h2.float()

        vm=key_valid.unsqueeze(-1).to(h2.dtype)
        tm=target_mask.unsqueeze(-1).to(h2.dtype)

        global_mean=(h2*vm).sum(1)/vm.sum(1).clamp(min=1)
        target_mean=(h2*tm).sum(1)/tm.sum(1).clamp(min=1)

        x=self.head_norm(
            torch.cat([target_mean,global_mean],dim=-1)
        )
        z=torch.cat([
            x,
            self.pos_mlp(pos.float()),
            self.domain_emb(domain_id),
        ],dim=-1)
        z=F.gelu(self.fc1(self.drop(z)))
        return self.classifier(self.drop(z))


student=KDStudent(
    backbone,NUM_LABELS,HEAD_DROPOUT
).to(DEVICE)

# Full student fine-tuning is intentional for KD.
assert all(p.requires_grad for p in student.backbone.parameters())

total_params=sum(p.numel() for p in student.parameters())
trainable_params=sum(p.numel() for p in student.parameters() if p.requires_grad)
print("Student total params:",f"{total_params/1e6:.3f}M")
print("Student trainable params:",f"{trainable_params/1e6:.3f}M")
print("Trainable %:",f"{100*trainable_params/total_params:.2f}%")


Aligned heldout teacher logits: (16845, 13) | teacher train argmax acc: 0.9628
Student total params: 81.553M
Student trainable params: 81.553M
Trainable %: 100.00%


In [27]:
# ============================================================
# CELL 11 — Dataloaders, layerwise-LR optimizer, KD losses
# ============================================================
class SortishBatchSampler(torch.utils.data.Sampler):
    def __init__(self,lengths,batch_size,seed=SEED,bucket_mult=LENGTH_BUCKET_MULT):
        self.lengths=list(map(int,lengths))
        self.batch_size=int(batch_size)
        self.seed=int(seed)
        self.bucket_size=max(
            self.batch_size,self.batch_size*int(bucket_mult)
        )
        self.epoch=0

    def set_epoch(self,epoch):
        self.epoch=int(epoch)

    def __len__(self):
        return math.ceil(len(self.lengths)/self.batch_size)

    def __iter__(self):
        rng=np.random.default_rng(self.seed+self.epoch)
        idx=np.arange(len(self.lengths),dtype=np.int64)
        rng.shuffle(idx)

        batches=[]
        for s in range(0,len(idx),self.bucket_size):
            chunk=idx[s:s+self.bucket_size]
            chunk=sorted(
                chunk.tolist(),
                key=lambda i:self.lengths[i],
                reverse=True
            )
            batches.extend(
                chunk[j:j+self.batch_size]
                for j in range(0,len(chunk),self.batch_size)
            )

        rng.shuffle(batches)
        yield from batches


def make_loader(records,labeled,batch_size,shuffle,teacher_logits=None):
    t0=time.time()
    ds=RRDataset(
        records,
        labeled=labeled,
        teacher_logits=teacher_logits,
    )
    print(
        f"tokenized {len(ds)} records in {time.time()-t0:.1f}s",
        "| teacher:",teacher_logits is not None
    )

    kw=dict(
        collate_fn=collate_rr,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type=="cuda"),
        persistent_workers=NUM_WORKERS>0,
    )

    if shuffle and LENGTH_GROUPED_TRAINING:
        return DataLoader(
            ds,
            batch_sampler=SortishBatchSampler(
                ds.lengths,batch_size,seed=SEED
            ),
            **kw,
        )

    if shuffle:
        g=torch.Generator(); g.manual_seed(SEED)
        return DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=True,
            generator=g,
            **kw,
        )

    order=sorted(
        range(len(ds)),
        key=lambda i:ds.lengths[i]
    )
    return DataLoader(
        Subset(ds,order),
        batch_size=batch_size,
        shuffle=False,
        **kw,
    )


train_loader=make_loader(
    train_records,True,MICRO_BATCH_SIZE,True,
    teacher_logits=teacher_train_logits,
)
dev_loader=make_loader(
    dev_records,True,EVAL_BATCH_SIZE,False,
    teacher_logits=None,
)


def make_student_param_groups(m):
    groups=[]
    n=len(m.backbone.blocks)

    # Lowest pretrained parameters move the least.
    emb_lr=STUDENT_BACKBONE_LR*(STUDENT_LAYERWISE_LR_DECAY**n)
    groups.append({
        "params":list(m.backbone.token_emb.parameters()),
        "lr":emb_lr,
        "weight_decay":WEIGHT_DECAY,
        "name":"token_emb",
    })

    for i,blk in enumerate(m.backbone.blocks):
        depth_from_top=(n-1)-i
        lr=STUDENT_BACKBONE_LR*(
            STUDENT_LAYERWISE_LR_DECAY**depth_from_top
        )
        groups.append({
            "params":list(blk.parameters()),
            "lr":lr,
            "weight_decay":WEIGHT_DECAY,
            "name":f"block_{i}",
        })

    groups.append({
        "params":list(m.backbone.final_ln.parameters()),
        "lr":STUDENT_BACKBONE_LR,
        "weight_decay":WEIGHT_DECAY,
        "name":"final_ln",
    })

    head_params=[]
    for name,p in m.named_parameters():
        if name.startswith(HEAD_PREFIXES):
            head_params.append(p)
    groups.append({
        "params":head_params,
        "lr":STUDENT_HEAD_LR,
        "weight_decay":WEIGHT_DECAY,
        "name":"rr_head",
    })
    return groups


param_groups=make_student_param_groups(student)
if DEVICE.type=="cuda":
    try:
        optimizer=torch.optim.AdamW(param_groups,fused=True)
        print("AdamW fused: True")
    except Exception as e:
        print("Fused AdamW unavailable:",repr(e)[:120])
        optimizer=torch.optim.AdamW(param_groups)
else:
    optimizer=torch.optim.AdamW(param_groups)

updates_per_epoch=math.ceil(
    len(train_loader)/GRAD_ACCUM_STEPS
)
total_updates=NUM_EPOCHS*updates_per_epoch
warmup_updates=max(
    1,int(total_updates*WARMUP_RATIO)
)

def lr_mult(step):
    if step<warmup_updates:
        return max(1e-8,step/warmup_updates)
    p=(step-warmup_updates)/max(
        1,total_updates-warmup_updates
    )
    return 0.5*(
        1+math.cos(math.pi*min(max(p,0.0),1.0))
    )

scheduler=torch.optim.lr_scheduler.LambdaLR(
    optimizer,lr_lambda=lr_mult
)

amp_enabled=bool(USE_AMP and DEVICE.type=="cuda")
scaler=torch.amp.GradScaler(
    "cuda",
    enabled=amp_enabled,
    init_scale=AMP_INIT_SCALE,
    growth_factor=AMP_GROWTH_FACTOR,
    backoff_factor=AMP_BACKOFF_FACTOR,
    growth_interval=AMP_GROWTH_INTERVAL,
)

def hard_loss_fn(student_logits,labels,weights):
    return F.cross_entropy(
        student_logits.float(),
        labels,
        weight=weights,
        label_smoothing=LABEL_SMOOTHING,
    )

def kd_kl_loss(
    student_logits,
    teacher_logits,
    labels,
    T=KD_TEMPERATURE,
    class_weights=None,
):
    s=student_logits.float()/T
    t=teacher_logits.float()/T

    log_ps=F.log_softmax(s,dim=-1)
    pt=F.softmax(t,dim=-1)

    per=F.kl_div(
        log_ps,pt,
        reduction="none",
    ).sum(-1)

    if KD_WEIGHT_BY_CLASS and class_weights is not None:
        wt=class_weights[labels].float()
        wt=wt/wt.mean().clamp_min(1e-8)
        per=per*wt

    return per.mean()*(T*T)


print(
    "Train batches/epoch:",len(train_loader),
    "| updates/epoch:",updates_per_epoch,
    "| total updates:",total_updates,
    "| warmup:",warmup_updates,
    "| effective batch:",MICRO_BATCH_SIZE*GRAD_ACCUM_STEPS,
    "| AMP:",amp_enabled,
)
print("AMP initial scale:", scaler.get_scale() if amp_enabled else "disabled")
print("Optimizer LRs:")
for g in optimizer.param_groups:
    print(
        f" {g.get('name','?'):12s}",
        f"{g['lr']:.3e}",
        f"params={sum(p.numel() for p in g['params'])/1e6:.3f}M"
    )


tokenized 16845 records in 2.7s | teacher: True
tokenized 2142 records in 0.3s | teacher: False
AdamW fused: True
Train batches/epoch: 8423 | updates/epoch: 1053 | total updates: 3159 | warmup: 189 | effective batch: 16 | AMP: True
AMP initial scale: 4096.0
Optimizer LRs:
 token_emb    1.353e-13 params=15.729M
 block_0      1.471e-13 params=4.924M
 block_1      1.599e-13 params=4.924M
 block_2      1.738e-13 params=4.924M
 block_3      1.889e-13 params=4.924M
 block_4      2.053e-13 params=4.924M
 block_5      2.231e-13 params=4.924M
 block_6      2.425e-13 params=4.924M
 block_7      2.636e-13 params=4.924M
 block_8      2.866e-13 params=4.924M
 block_9      3.115e-13 params=4.924M
 block_10     3.386e-13 params=4.924M
 block_11     3.680e-13 params=4.924M
 block_12     4.000e-13 params=4.924M
 final_ln     4.000e-13 params=0.001M
 rr_head      7.000e-12 params=1.817M


In [28]:
# ============================================================
# CELL 12 — Metrics and evaluation
# ============================================================
def _forward_batch(m,batch):
    return m(batch['input_ids'].to(DEVICE,non_blocking=True),batch['key_valid'].to(DEVICE,non_blocking=True),
             batch['target_mask'].to(DEVICE,non_blocking=True),batch['pos'].to(DEVICE,non_blocking=True),batch['domain_id'].to(DEVICE,non_blocking=True))

@torch.no_grad()
def predict_loader(m,loader):
    m.eval(); yt=[]; yp=[]; meta=[]
    for batch in loader:
        with torch.amp.autocast(device_type='cuda',dtype=torch.float16,enabled=amp_enabled): logits=_forward_batch(m,batch)
        yp.extend(logits.float().argmax(-1).cpu().tolist())
        if 'labels' in batch: yt.extend(batch['labels'].tolist())
        meta.extend(zip(batch['doc_id'],batch['sent_idx'],batch['domain']))
    return yt,yp,meta

def score_ids(y_true,y_pred,title='Combined'):
    ids=list(range(NUM_LABELS)); f=f1_score(y_true,y_pred,labels=ids,average='macro',zero_division=0); p=precision_score(y_true,y_pred,labels=ids,average='macro',zero_division=0); r=recall_score(y_true,y_pred,labels=ids,average='macro',zero_division=0)
    print(f"{title}: F1={f:.6f} P={p:.6f} R={r:.6f}"); return f,p,r

def detailed_dev_eval(m,verbose=True):
    yt,yp,meta=predict_loader(m,dev_loader); combined=score_ids(yt,yp,'Combined CL+IT')
    _pc0=Counter(yp)
    if max(_pc0.values())/max(1,len(yp))>0.60: print('   ⚠️ >60% of predictions are one class — possible collapse:',{LABELS[k]:v for k,v in _pc0.items()})
    for dom in ['CL','IT']:
        idx=[i for i,x in enumerate(meta) if x[2]==dom]; score_ids([yt[i] for i in idx],[yp[i] for i in idx],dom)
    if not verbose: return combined,yt,yp,meta
    present=sorted(set(yt)); absent=[i for i in range(NUM_LABELS) if i not in present]
    print(f"Dev contains {len(present)}/{NUM_LABELS} labels; formal Macro-F1 ceiling from label absence = {len(present)/NUM_LABELS:.6f}")
    if absent: print('Absent dev labels:',[LABELS[i] for i in absent])
    pc,gc=Counter(yp),Counter(yt); fp=f1_score(yt,yp,labels=present,average='macro',zero_division=0)
    print(f"   Macro-F1 over {len(present)} labels present in dev: {fp:.6f}")
    print('   gold counts:',{LABELS[k]:gc[k] for k in present}); print('   pred counts:',{LABELS[k]:pc[k] for k in sorted(pc)})
    print("\nPer-label report (all 13 labels explicitly included):")
    print(classification_report(yt,yp,labels=list(range(NUM_LABELS)),target_names=LABELS,digits=4,zero_division=0))
    return combined,yt,yp,meta

@torch.no_grad()
def predict_logits(m,loader):
    m.eval(); outs=[]; meta=[]; yt=[]
    for batch in loader:
        with torch.amp.autocast(device_type='cuda',dtype=torch.float16,enabled=amp_enabled): lg=_forward_batch(m,batch)
        outs.append(lg.float().cpu().numpy()); meta.extend(zip(batch['doc_id'],batch['sent_idx'],batch['domain']))
        if 'labels' in batch: yt.extend(batch['labels'].tolist())
    return np.concatenate(outs,0),meta,yt


In [29]:
# ============================================================
# CELL 13 — Knowledge Distillation training + mid-epoch dev selection
# ============================================================
def fmt_sec(s):
    s=max(0,int(s))
    h,s=divmod(s,3600)
    m,s=divmod(s,60)
    return f"{h:02d}:{m:02d}:{s:02d}"

def student_state_dict(m):
    return {
        k:v.detach().cpu()
        for k,v in m.state_dict().items()
    }

best_f1=-1.0
best_epoch=None
best_frac=None
best_batch_in_epoch=None
best_global_update=None
bad_evals=0
global_update=0
amp_overflows=0
history=[]
top_snapshots=[]
train_start=time.time()

optimizer.zero_grad(set_to_none=True)

for epoch in range(1,NUM_EPOCHS+1):
    if hasattr(train_loader.batch_sampler,"set_epoch"):
        train_loader.batch_sampler.set_epoch(epoch)

    student.train()
    run_total=0.0
    run_hard=0.0
    run_kd=0.0
    seen_batches=0
    ep0=time.time()

    # Align evaluation points to gradient-accumulation boundaries.
    raw_eval=[
        int(round(len(train_loader)*j/EVAL_PER_EPOCH))
        for j in range(1,EVAL_PER_EPOCH+1)
    ]
    eval_steps=[]
    for s in raw_eval:
        s=max(GRAD_ACCUM_STEPS,min(len(train_loader),s))
        if s<len(train_loader):
            s=max(GRAD_ACCUM_STEPS,(s//GRAD_ACCUM_STEPS)*GRAD_ACCUM_STEPS)
        else:
            s=len(train_loader)
        eval_steps.append(s)
    eval_steps=sorted(set(eval_steps))

    if epoch==1:
        print("Dev evaluation at batches:",eval_steps,"of",len(train_loader))

    stop_training=False

    for step,batch in enumerate(train_loader,1):
        labels=batch["labels"].to(
            DEVICE,non_blocking=True
        )
        teacher_logits=batch["teacher_logits"].to(
            DEVICE,non_blocking=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=amp_enabled,
        ):
            student_logits=_forward_batch(
                student,batch
            )
            hard=hard_loss_fn(
                student_logits,labels,class_w
            )
            soft=kd_kl_loss(
                student_logits,
                teacher_logits,
                labels,
                KD_TEMPERATURE,
                class_w,
            )
            loss=(
                HARD_LOSS_WEIGHT*hard
                + KD_LOSS_WEIGHT*soft
            )/GRAD_ACCUM_STEPS

        if step%NAN_CHECK_EVERY==0 or step==len(train_loader):
            assert bool(torch.isfinite(loss.detach()).item()),(
                epoch,step,loss
            )

        scaler.scale(loss).backward()

        run_total+=float(loss.detach().float().cpu())*GRAD_ACCUM_STEPS
        run_hard+=float(hard.detach().float().cpu())
        run_kd+=float(soft.detach().float().cpu())
        seen_batches+=1

        if step%GRAD_ACCUM_STEPS==0 or step==len(train_loader):
            # GradScaler records Inf/NaN during unscale_. If overflow occurred,
            # scaler.step() safely skips optimizer.step() and scaler.update()
            # lowers the scale. Do NOT crash the whole training run.
            scaler.unscale_(optimizer)
            grad_norm=torch.nn.utils.clip_grad_norm_(
                [p for p in student.parameters() if p.requires_grad],
                GRAD_CLIP,
            )
            grad_is_finite=bool(
                torch.isfinite(torch.as_tensor(grad_norm)).item()
            )

            old_scale=float(scaler.get_scale())
            scaler.step(optimizer)
            scaler.update()
            new_scale=float(scaler.get_scale())
            optimizer.zero_grad(set_to_none=True)

            update_succeeded=(new_scale>=old_scale) and grad_is_finite
            if update_succeeded:
                scheduler.step()
                global_update+=1
            else:
                amp_overflows+=1
                print(
                    f"⚠️ AMP overflow handled at epoch={epoch} batch={step} "
                    f"| grad_norm={float(torch.as_tensor(grad_norm))} "
                    f"| scale {old_scale:.0f} -> {new_scale:.0f} "
                    f"| optimizer update skipped "
                    f"| total_overflows={amp_overflows}",
                    flush=True,
                )

        if step%PRINT_EVERY==0 or step==len(train_loader):
            elapsed=time.time()-ep0
            rate=elapsed/max(step,1)
            eta_ep=rate*(len(train_loader)-step)
            completed=(epoch-1)+step/len(train_loader)
            eta_total=rate*len(train_loader)*max(
                0,NUM_EPOCHS-completed
            )
            print(
                f"epoch={epoch} batch={step}/{len(train_loader)} "
                f"loss={run_total/max(seen_batches,1):.4f} "
                f"hard={run_hard/max(seen_batches,1):.4f} "
                f"kd={run_kd/max(seen_batches,1):.4f} "
                f"top_bb_lr={optimizer.param_groups[-2]['lr']:.2e} "
                f"head_lr={optimizer.param_groups[-1]['lr']:.2e} "
                f"elapsed={fmt_sec(elapsed)} "
                f"ETA_epoch={fmt_sec(eta_ep)} "
                f"ETA_maxtrain≈{fmt_sec(eta_total)}",
                flush=True,
            )

        if step in eval_steps:
            metrics,y_true,y_pred,meta=detailed_dev_eval(
                student,verbose=False
            )
            macro_f1,macro_p,macro_r=metrics
            frac=(epoch-1)+step/len(train_loader)

            row={
                "epoch":epoch,
                "epoch_frac":float(frac),
                "batch_in_epoch":int(step),
                "global_update":int(global_update),
                "train_loss":run_total/max(seen_batches,1),
                "hard_loss":run_hard/max(seen_batches,1),
                "kd_loss":run_kd/max(seen_batches,1),
                "macro_f1":float(macro_f1),
                "macro_precision":float(macro_p),
                "macro_recall":float(macro_r),
                "elapsed_sec":time.time()-ep0,
                "amp_overflows":int(amp_overflows),
                "amp_scale":float(scaler.get_scale()),
            }
            history.append(row)
            print("KD eval summary:",row)

            # Save top-k heldout student snapshots for test-time ensembling.
            qualifies=(
                len(top_snapshots)<TOPK_HELDOUT_SNAPSHOTS
                or macro_f1>min(x["f1"] for x in top_snapshots)
            )
            snapshot_state=None
            if qualifies:
                snapshot_state=student_state_dict(student)
                sp=ROOT/f"kd_snapshot_e{frac:.3f}_f1_{macro_f1:.6f}.pt"
                torch.save({
                    "student_state":snapshot_state,
                    "epoch_frac":float(frac),
                    "macro_f1":float(macro_f1),
                    "batch_in_epoch":int(step),
                    "global_update":int(global_update),
                },sp)

                top_snapshots.append({
                    "f1":float(macro_f1),
                    "frac":float(frac),
                    "path":str(sp),
                })
                top_snapshots.sort(
                    key=lambda x:x["f1"],
                    reverse=True,
                )
                while len(top_snapshots)>TOPK_HELDOUT_SNAPSHOTS:
                    drop=top_snapshots.pop(-1)
                    try:
                        os.remove(drop["path"])
                    except FileNotFoundError:
                        pass

                print(
                    "Top KD snapshots:",
                    [
                        (round(x["f1"],6),round(x["frac"],3))
                        for x in top_snapshots
                    ]
                )

            if macro_f1>best_f1:
                best_f1=float(macro_f1)
                best_epoch=int(math.ceil(frac))
                best_frac=float(frac)
                best_batch_in_epoch=int(step)
                best_global_update=int(global_update)
                bad_evals=0

                if snapshot_state is None:
                    snapshot_state=student_state_dict(student)

                torch.save({
                    "student_state":snapshot_state,
                    "student_phase1_checkpoint":str(PHASE1_CKPT),
                    "teacher_logits_path":str(TEACHER_TRAIN_PATH),
                    "best_epoch":best_epoch,
                    "best_frac":best_frac,
                    "best_batch_in_epoch":best_batch_in_epoch,
                    "best_global_update":best_global_update,
                    "best_macro_f1":best_f1,
                    "amp_overflows":int(amp_overflows),
                    "amp_scale":float(scaler.get_scale()),
                    "history":history,
                    "kd_temperature":KD_TEMPERATURE,
                    "hard_loss_weight":HARD_LOSS_WEIGHT,
                    "kd_loss_weight":KD_LOSS_WEIGHT,
                    "layerwise_lr_decay":STUDENT_LAYERWISE_LR_DECAY,
                    "labels":LABELS,
                    "label_to_id":LABEL_TO_ID,
                    "experiment":"Phase3 Exp6 KD rsLoRA teacher -> Model2 student",
                },BEST_CKPT)

                print(
                    f"✅ NEW BEST KD STUDENT at epoch {frac:.3f} "
                    f"| batch={step} | update={global_update} "
                    f"| F1={best_f1:.6f}"
                )
            else:
                bad_evals+=1

            print(
                f"Best KD so far: epoch_frac={best_frac:.3f} "
                f"Macro-F1={best_f1:.6f} "
                f"| non-improving evals={bad_evals}\n"
            )

            if bad_evals>=EARLY_STOP_PATIENCE_EVALS:
                print(
                    f"⏹️ Early stopping after {bad_evals} "
                    "non-improving dev evaluations."
                )
                stop_training=True
                break

            student.train()

    if stop_training:
        break

with open(ROOT/"history.json","w",encoding="utf-8") as f:
    json.dump(history,f,indent=2)
with open(ROOT/"top_snapshots.json","w",encoding="utf-8") as f:
    json.dump(top_snapshots,f,indent=2)

assert best_frac is not None
print("KD training complete | elapsed:",fmt_sec(time.time()-train_start))
print(
    "BEST_EPOCH =",best_epoch,
    "| BEST_EPOCH_FRAC =",best_frac,
    "| BEST_BATCH =",best_batch_in_epoch,
    "| BEST_UPDATE =",best_global_update,
    "| BEST_MACRO_F1 =",best_f1,
    "| AMP_OVERFLOWS =",amp_overflows,
    "| FINAL_AMP_SCALE =",scaler.get_scale()
)


Dev evaluation at batches: [2104, 4208, 6312, 8423] of 8423
epoch=1 batch=250/8423 loss=2.9836 hard=2.3820 kd=3.3847 top_bb_lr=6.56e-06 head_lr=1.15e-04 elapsed=00:00:15 ETA_epoch=00:08:32 ETA_maxtrain≈00:26:07
epoch=1 batch=500/8423 loss=2.7243 hard=2.2021 kd=3.0725 top_bb_lr=1.31e-05 head_lr=2.30e-04 elapsed=00:00:31 ETA_epoch=00:08:13 ETA_maxtrain≈00:25:43
epoch=1 batch=750/8423 loss=2.5632 hard=2.0812 kd=2.8846 top_bb_lr=1.97e-05 head_lr=3.44e-04 elapsed=00:00:46 ETA_epoch=00:07:57 ETA_maxtrain≈00:25:26
epoch=1 batch=1000/8423 loss=2.4670 hard=2.0128 kd=2.7699 top_bb_lr=2.65e-05 head_lr=4.63e-04 elapsed=00:01:02 ETA_epoch=00:07:40 ETA_maxtrain≈00:25:05
epoch=1 batch=1250/8423 loss=2.3851 hard=1.9644 kd=2.6656 top_bb_lr=3.30e-05 head_lr=5.78e-04 elapsed=00:01:17 ETA_epoch=00:07:22 ETA_maxtrain≈00:24:41
epoch=1 batch=1500/8423 loss=2.3190 hard=1.9163 kd=2.5875 top_bb_lr=3.96e-05 head_lr=6.93e-04 elapsed=00:01:32 ETA_epoch=00:07:05 ETA_maxtrain≈00:24:21
epoch=1 batch=1750/8423 loss=2.

In [30]:
# ============================================================
# CELL 14 — Reload BEST KD student and reconstruct dev prediction.json
# ============================================================
best_obj=torch.load(
    BEST_CKPT,map_location="cpu",weights_only=False
)

# Rebuild a clean Phase-1 Model2 student.
def build_fresh_student():
    seed_everything(SEED)
    bb=build_backbone()
    o=torch.load(
        PHASE1_CKPT,
        map_location="cpu",
        weights_only=False,
    )
    st=o["model"] if isinstance(o,dict) and "model" in o else o
    bb.load_state_dict(st,strict=True)
    for p in bb.parameters():
        p.requires_grad=True
    return KDStudent(
        bb,NUM_LABELS,HEAD_DROPOUT
    ).to(DEVICE)

student=build_fresh_student()
student.load_state_dict(
    best_obj["student_state"],
    strict=True,
)

best_f1=float(best_obj["best_macro_f1"])
best_epoch=int(best_obj["best_epoch"])
best_frac=float(best_obj.get("best_frac",best_epoch))

print(
    "Reloaded best KD student:",
    "epoch_frac=",best_frac,
    "Macro-F1=",best_f1
)

dev_logits,dev_meta,dev_y=predict_logits(
    student,dev_loader
)
np.savez_compressed(
    ROOT/"dev_logits.npz",
    logits=dev_logits,
    y_true=np.asarray(dev_y,dtype=np.int64),
    doc_id=np.asarray([x[0] for x in dev_meta],dtype=object),
    sent_idx=np.asarray([x[1] for x in dev_meta],dtype=np.int64),
    domain=np.asarray([x[2] for x in dev_meta],dtype=object),
)

def records_to_prediction(meta,pred_ids):
    by_doc={}
    for (did,si,dom),pid in zip(meta,pred_ids):
        by_doc.setdefault(did,[]).append(
            (si,LABELS[int(pid)])
        )
    out={}
    for did,items in by_doc.items():
        items.sort(key=lambda x:x[0])
        out[did]=[lab for _,lab in items]
    return out

dev_pred=records_to_prediction(
    dev_meta,dev_logits.argmax(-1)
)
with open(DEV_PRED,"w",encoding="utf-8") as f:
    json.dump(dev_pred,f,ensure_ascii=False,indent=2)

def write_truth(docs,path):
    truth={}
    for d in docs:
        did=normalize_id(d["id"])
        truth[did]=list(d["labels"])
    with open(path,"w",encoding="utf-8") as f:
        json.dump(truth,f,ensure_ascii=False,indent=2)

write_truth(cl_dev_raw,CL_DEV_TRUTH)
write_truth(it_dev_raw,IT_DEV_TRUTH)
with open(LABELS_JSON,"w",encoding="utf-8") as f:
    json.dump(LABELS,f,ensure_ascii=False,indent=2)

for p in [CL_DEV_TRUTH,IT_DEV_TRUTH,LABELS_JSON,DEV_PRED]:
    assert Path(p).is_file(),p

print("✅ KD DEV prediction/truth/labels written.")


Reloaded best KD student: epoch_frac= 1.24979223554553 Macro-F1= 0.3996394352517635
✅ KD DEV prediction/truth/labels written.


In [31]:
# ============================================================
# CELL 15 — EXACT local Phase-3 evaluator script supplied by course
# ============================================================

EVALUATOR_CODE = r'''#!/usr/bin/env python3
"""Local evaluator for Phase 3.

The competition scorer evaluates two gold/reference JSON files together and
one participant prediction.json. This local evaluator mirrors that setup
using the student's own held-out splits.
"""
import argparse
import json
import sys

from sklearn.metrics import f1_score, precision_score, recall_score

EXPECTED_LABEL_COUNT = 13


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def normalize_id(doc_id):
    if not isinstance(doc_id, str) or not doc_id.strip():
        raise ValueError("Document IDs must be non-empty strings.")
    return doc_id[:-4] if doc_id.endswith(".txt") else doc_id


def load_vocab(data):
    if isinstance(data, list):
        labels = data
    elif isinstance(data, dict) and data:
        values = list(data.values())
        if all(isinstance(v, int) and not isinstance(v, bool) for v in values):
            labels = list(data.keys())
        elif all(isinstance(v, str) for v in values):
            labels = values
        else:
            raise ValueError("Unsupported labels.json format.")
    else:
        raise ValueError("labels.json must be a non-empty list or object.")

    if len(labels) != EXPECTED_LABEL_COUNT:
        raise ValueError(
            f"Expected {EXPECTED_LABEL_COUNT} labels, found {len(labels)}."
        )
    if any(not isinstance(label, str) or not label.strip() for label in labels):
        raise ValueError("Labels must be unique non-empty strings.")
    if len(set(labels)) != len(labels):
        raise ValueError("Labels must be unique non-empty strings.")
    return labels


def normalize_reference(data, source_name):
    if not isinstance(data, dict):
        raise ValueError(f"{source_name}: reference data must be a JSON object.")

    normalized = {}
    for raw_id, labels in data.items():
        doc_id = normalize_id(raw_id)
        if doc_id in normalized:
            raise ValueError(
                f"{source_name}: duplicate document ID after .txt normalization: {doc_id}"
            )
        normalized[doc_id] = labels
    return normalized


def normalize_predictions(data):
    if not isinstance(data, dict):
        raise ValueError("prediction.json must be a JSON object mapping document IDs to label lists.")

    normalized = {}
    for raw_id, labels in data.items():
        doc_id = normalize_id(raw_id)
        if doc_id in normalized:
            raise ValueError(
                f"prediction.json: duplicate document ID after .txt normalization: {doc_id}"
            )
        normalized[doc_id] = labels
    return normalized


def main():
    parser = argparse.ArgumentParser(
        description=(
            "Evaluate Phase 3 sentence-label predictions on two local held-out "
            "reference splits, mirroring the competition's combined scoring setup."
        )
    )
    parser.add_argument("--truth-cl", required=True)
    parser.add_argument("--truth-it", required=True)
    parser.add_argument("--prediction", required=True)
    parser.add_argument("--labels", required=True)
    args = parser.parse_args()

    truth_cl = normalize_reference(load_json(args.truth_cl), "--truth-cl")
    truth_it = normalize_reference(load_json(args.truth_it), "--truth-it")
    predictions = normalize_predictions(load_json(args.prediction))
    labels = load_vocab(load_json(args.labels))
    allowed = set(labels)

    overlap = set(truth_cl) & set(truth_it)
    if overlap:
        sample = sorted(overlap)[:5]
        suffix = "..." if len(overlap) > 5 else ""
        raise ValueError(
            "The two reference splits contain overlapping document IDs: "
            + ", ".join(sample)
            + suffix
        )

    truth = {}
    truth.update(truth_cl)
    truth.update(truth_it)

    if set(truth) != set(predictions):
        missing = sorted(set(truth) - set(predictions))
        extra = sorted(set(predictions) - set(truth))
        parts = [f"missing={len(missing)}", f"extra={len(extra)}"]
        if missing:
            parts.append(f"example missing ID(s)={missing[:5]}")
        if extra:
            parts.append(f"example extra ID(s)={extra[:5]}")
        raise ValueError("ID mismatch: " + "; ".join(parts))

    y_true = []
    y_pred = []

    for doc_id in truth:
        gold = truth[doc_id]
        pred = predictions[doc_id]

        if not isinstance(gold, list) or not isinstance(pred, list):
            raise ValueError(f"{doc_id}: labels must be JSON lists.")
        if len(gold) != len(pred):
            raise ValueError(
                f"{doc_id}: sentence-count mismatch ({len(gold)} vs {len(pred)})."
            )
        if any(label not in allowed for label in gold + pred):
            raise ValueError(f"{doc_id}: unknown label encountered.")

        y_true.extend(gold)
        y_pred.extend(pred)

    score = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    macro_precision = precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    macro_recall = recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)

    print(f"CL documents: {len(truth_cl)}")
    print(f"IT documents: {len(truth_it)}")
    print(f"Total documents: {len(truth)}")
    print(f"Total sentences: {len(y_true)}")
    print(f"Macro-F1:        {score:.6f}")
    print(f"Macro-Precision: {macro_precision:.6f}")
    print(f"Macro-Recall:    {macro_recall:.6f}")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(f"EVALUATION FAILED: {exc}", file=sys.stderr)
        sys.exit(1)
'''

with open(EVAL_SCRIPT, "w", encoding="utf-8") as f:
    f.write(EVALUATOR_CODE)

print("Saved exact evaluator to:", EVAL_SCRIPT)


Saved exact evaluator to: /kaggle/working/phase3_exp6_kd_rslora_teacher_model2_student/evaluate_phase3.py


In [32]:
# ============================================================
# CELL 16 — RUN THE OFFICIAL-STYLE LOCAL EVALUATOR
# Do not continue to final refit until this passes.
# ============================================================

cmd = [
    sys.executable,
    str(EVAL_SCRIPT),
    "--truth-cl", str(CL_DEV_TRUTH),
    "--truth-it", str(IT_DEV_TRUTH),
    "--prediction", str(DEV_PRED),
    "--labels", str(LABELS_JSON),
]

print("Running:")
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print("\nSTDOUT:\n", result.stdout)
if result.stderr:
    print("\nSTDERR:\n", result.stderr)

assert result.returncode == 0, "❌ Phase-3 evaluator failed."
print("✅ DEV prediction passed the exact Phase-3 evaluator.")


Running:
/usr/bin/python3 /kaggle/working/phase3_exp6_kd_rslora_teacher_model2_student/evaluate_phase3.py --truth-cl /kaggle/working/phase3_exp6_kd_rslora_teacher_model2_student/cl_dev_truth.json --truth-it /kaggle/working/phase3_exp6_kd_rslora_teacher_model2_student/it_dev_truth.json --prediction /kaggle/working/phase3_exp6_kd_rslora_teacher_model2_student/prediction_dev.json --labels /kaggle/working/phase3_exp6_kd_rslora_teacher_model2_student/labels.json

STDOUT:
 CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.399639
Macro-Precision: 0.448664
Macro-Recall:    0.380026

✅ DEV prediction passed the exact Phase-3 evaluator.


In [34]:
# ============================================================
# CELL 17 — Optional transition-aware decoding tuned on DEV
#
# Improvements:
#   * CL and IT transition/prior statistics estimated separately
#   * additive smoothing
#   * no hard "impossible label" bans
#   * only adopt decoding when DEV Macro-F1 improves by >= 0.005
# ============================================================

from sklearn.metrics import f1_score as _f1

def transition_matrix(docs_list, smooth=1.0):
    T = np.full((NUM_LABELS, NUM_LABELS), float(smooth), dtype=np.float64)
    for d in docs_list:
        ids = [LABEL_TO_ID[x] for x in d["labels"]]
        for a, b in zip(ids[:-1], ids[1:]):
            T[a, b] += 1.0
    T /= T.sum(1, keepdims=True)
    return np.log(T)

def class_log_prior(docs_list, smooth=1.0):
    c = np.full(NUM_LABELS, float(smooth), dtype=np.float64)
    for d in docs_list:
        for x in d["labels"]:
            c[LABEL_TO_ID[x]] += 1.0
    return np.log(c / c.sum())

def log_softmax_np(x):
    x = x - x.max(-1, keepdims=True)
    return x - np.log(np.exp(x).sum(-1, keepdims=True))

def viterbi_decode(logp, trans, beta):
    n, K = logp.shape
    score = logp[0].copy()
    back = np.zeros((n, K), dtype=np.int32)
    tr = beta * trans

    for t in range(1, n):
        cand = score[:, None] + tr
        back[t] = cand.argmax(0)
        score = cand[back[t], np.arange(K)] + logp[t]

    path = np.zeros(n, dtype=np.int64)
    path[-1] = int(score.argmax())

    for t in range(n - 1, 0, -1):
        path[t - 1] = back[t, path[t]]

    return path

def build_domain_stats(cl_docs, it_docs):
    return {
        "CL": {
            "trans": transition_matrix(cl_docs, smooth=1.0),
            "prior": class_log_prior(cl_docs, smooth=1.0),
        },
        "IT": {
            "trans": transition_matrix(it_docs, smooth=1.0),
            "prior": class_log_prior(it_docs, smooth=1.0),
        },
    }

def decode_docs(logits, meta, domain_stats, alpha, beta):
    # logits aligned with meta=(doc_id, sent_idx, domain)
    by_doc = {}
    for i, (did, si, dom) in enumerate(meta):
        by_doc.setdefault(did, []).append((si, i, dom))

    pred = np.zeros(len(meta), dtype=np.int64)

    for did, items in by_doc.items():
        items.sort(key=lambda x: x[0])
        idx = [i for _, i, _ in items]
        dom = items[0][2]
        stats = domain_stats[dom]

        lp = (
            log_softmax_np(logits[idx].astype(np.float64))
            - alpha * stats["prior"][None, :]
        )

        pred[idx] = (
            viterbi_decode(lp, stats["trans"], beta)
            if beta > 0
            else lp.argmax(-1)
        )

    return pred

# Tune only on held-out DEV predictions.
dev_logits, dev_meta, dev_y = predict_logits(student, dev_loader)
np.savez_compressed(
    ROOT / "dev_logits.npz",
    logits=dev_logits,
    y=np.asarray(dev_y, dtype=np.int64),
    doc_id=np.asarray([x[0] for x in dev_meta]),
    sent_idx=np.asarray([x[1] for x in dev_meta], dtype=np.int64),
    domain=np.asarray([x[2] for x in dev_meta]),
)

domain_stats_train = build_domain_stats(
    cl_train_raw,
    it_train_raw,
)

def dev_f1(alpha, beta):
    p = decode_docs(
        dev_logits,
        dev_meta,
        domain_stats_train,
        alpha,
        beta
    )
    return _f1(
        dev_y,
        p,
        labels=list(range(NUM_LABELS)),
        average="macro",
        zero_division=0
    )

rows = []
for alpha in [-0.25, 0.0, 0.25, 0.5, 0.75]:
    for beta in [0.0, 0.25, 0.5, 1.0]:
        rows.append((dev_f1(alpha, beta), alpha, beta))

base_f1 = dev_f1(0.0, 0.0)
rows.sort(key=lambda r: (-r[0], r[2], abs(r[1])))

print(f"Baseline argmax (alpha=0, beta=0): {base_f1:.6f}")
print("Top decoding settings (dev F1, alpha, beta):")
for r in rows[:8]:
    print(f"  {r[0]:.6f}  alpha={r[1]:+.2f}  beta={r[2]:.2f}")

if rows[0][0] - base_f1 >= 0.005:
    DECODE_ALPHA, DECODE_BETA = rows[0][1], rows[0][2]
else:
    DECODE_ALPHA, DECODE_BETA = 0.0, 0.0

print("SELECTED decoding: alpha=", DECODE_ALPHA, "beta=", DECODE_BETA)

dec_ids = decode_docs(
    dev_logits,
    dev_meta,
    domain_stats_train,
    DECODE_ALPHA,
    DECODE_BETA
)

dev_pred_dec = records_to_prediction(dev_meta, dec_ids)
DEV_PRED_DEC = ROOT / "prediction_dev_decoded.json"

with open(DEV_PRED_DEC, "w", encoding="utf-8") as f:
    json.dump(dev_pred_dec, f, ensure_ascii=False, indent=2)

# Run the exact supplied evaluator again on the decoded DEV prediction.
res = subprocess.run(
    [
        sys.executable,
        str(EVAL_SCRIPT),
        "--truth-cl", str(CL_DEV_TRUTH),
        "--truth-it", str(IT_DEV_TRUTH),
        "--prediction", str(DEV_PRED_DEC),
        "--labels", str(LABELS_JSON),
    ],
    capture_output=True,
    text=True
)

print(res.stdout)
if res.stderr:
    print(res.stderr)

assert res.returncode == 0, "Decoded DEV prediction failed the exact evaluator."
print("✅ Decoded DEV prediction passed the exact Phase-3 evaluator.")


Baseline argmax (alpha=0, beta=0): 0.399639
Top decoding settings (dev F1, alpha, beta):
  0.415334  alpha=+0.25  beta=0.25
  0.411578  alpha=+0.25  beta=0.00
  0.410870  alpha=+0.50  beta=1.00
  0.408631  alpha=+0.25  beta=1.00
  0.400111  alpha=+0.25  beta=0.50
  0.400067  alpha=+0.50  beta=0.50
  0.399639  alpha=+0.00  beta=0.00
  0.394565  alpha=+0.00  beta=0.50
SELECTED decoding: alpha= 0.25 beta= 0.25
CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.415334
Macro-Precision: 0.469319
Macro-Recall:    0.391391

✅ Decoded DEV prediction passed the exact Phase-3 evaluator.


In [35]:
# ============================================================
# CELL 18 — Fresh Phase-1 Model2 KD student for train+dev refit
# ============================================================
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

BEST_EPOCH=int(best_obj["best_epoch"])
BEST_FRAC=float(best_obj.get("best_frac",BEST_EPOCH))
BEST_BATCH=int(best_obj.get("best_batch_in_epoch",0))
BEST_UPDATE=int(best_obj.get("best_global_update",0))

assert BEST_EPOCH>=1 and 0.0<BEST_FRAC<=NUM_EPOCHS

print(
    "Final KD refit target:",
    f"epoch_frac={BEST_FRAC:.3f}",
    f"| selected_batch={BEST_BATCH}",
    f"| selected_update={BEST_UPDATE}",
)

all_labeled_records=(
    make_labeled_records(cl_train_raw,"CL")
    +make_labeled_records(it_train_raw,"IT")
    +make_labeled_records(cl_dev_raw,"CL")
    +make_labeled_records(it_dev_raw,"IT")
)

# Build strongest leakage-safe final-refit teacher:
# TRAIN = heldout+refit teacher ensemble
# DEV   = exact ensemble if heldout DEV logits exist, else refit teacher only.
teacher_all_logits,FINAL_REFIT_TEACHER_MODE=build_final_refit_teacher_targets(
    train_records,
    dev_records,
)
assert teacher_all_logits.shape==(len(all_labeled_records),NUM_LABELS)
# Row order of the concatenated teacher targets must equal all_labeled_records.
_ord=[(r["doc_id"],r["sent_idx"],r["domain"]) for r in train_records+dev_records]
_all=[(r["doc_id"],r["sent_idx"],r["domain"]) for r in all_labeled_records]
assert _ord==_all,"train+dev record order differs from all_labeled_records"
assert np.array_equal(
    np.asarray([r["label_id"] for r in train_records+dev_records]),
    np.asarray([r["label_id"] for r in all_labeled_records]))

print(
    "Final refit teacher targets:",
    teacher_all_logits.shape,
    "| mode:",FINAL_REFIT_TEACHER_MODE
)


all_counts=Counter(
    x["label"] for x in all_labeled_records
)
N_all=len(all_labeled_records)
raw=np.array([
    math.sqrt(
        N_all/(NUM_LABELS*max(all_counts[lab],1))
    )
    for lab in LABELS
],dtype=np.float32)
raw/=raw.mean()
class_w_all=torch.tensor(
    np.clip(
        raw,CLASS_WEIGHT_MIN,CLASS_WEIGHT_MAX
    ),
    dtype=torch.float32,
    device=DEVICE,
)

# Save exactly what the final KD refit uses.
np.savez_compressed(
    ROOT/"teacher_logits_train_dev_ensemble_for_kd.npz",
    logits=teacher_all_logits.astype(np.float32),
    y=np.asarray(
        [r["label_id"] for r in all_labeled_records],
        dtype=np.int64,
    ),
    doc_id=np.asarray(
        [r["doc_id"] for r in all_labeled_records],
        dtype=object,
    ),
    sent_idx=np.asarray(
        [r["sent_idx"] for r in all_labeled_records],
        dtype=np.int64,
    ),
    domain=np.asarray(
        [r["domain"] for r in all_labeled_records],
        dtype=object,
    ),
    mode=np.asarray([FINAL_REFIT_TEACHER_MODE],dtype=object),
)
print(
    "✅ Saved final KD teacher targets:",
    ROOT/"teacher_logits_train_dev_ensemble_for_kd.npz"
)

final_student=build_fresh_student()

all_loader=make_loader(
    all_labeled_records,
    True,
    MICRO_BATCH_SIZE,
    True,
    teacher_logits=teacher_all_logits,
)

final_groups=make_student_param_groups(
    final_student
)
try:
    final_optimizer=torch.optim.AdamW(
        final_groups,
        fused=(DEVICE.type=="cuda"),
    )
except Exception:
    final_optimizer=torch.optim.AdamW(
        final_groups
    )

REFIT_TOTAL_BATCHES=int(
    round(
        BEST_FRAC
        *len(all_loader)
        /GRAD_ACCUM_STEPS
    )
)*GRAD_ACCUM_STEPS
REFIT_TOTAL_BATCHES=max(
    GRAD_ACCUM_STEPS,
    REFIT_TOTAL_BATCHES,
)

final_updates_per_epoch=math.ceil(
    len(all_loader)/GRAD_ACCUM_STEPS
)
SCHED_EPOCHS=(
    NUM_EPOCHS
    if REFIT_MATCH_ORIGINAL_SCHEDULE
    else max(1,math.ceil(BEST_FRAC))
)
final_total_updates=SCHED_EPOCHS*final_updates_per_epoch
final_warmup=max(
    1,int(final_total_updates*WARMUP_RATIO)
)

def final_lr_mult(step):
    if step<final_warmup:
        return max(1e-8,step/final_warmup)
    p=(step-final_warmup)/max(
        1,final_total_updates-final_warmup
    )
    return 0.5*(
        1+math.cos(math.pi*min(max(p,0.0),1.0))
    )

final_scheduler=torch.optim.lr_scheduler.LambdaLR(
    final_optimizer,
    lr_lambda=final_lr_mult,
)
final_scaler=torch.amp.GradScaler(
    "cuda",
    enabled=amp_enabled,
    init_scale=AMP_INIT_SCALE,
    growth_factor=AMP_GROWTH_FACTOR,
    backoff_factor=AMP_BACKOFF_FACTOR,
    growth_interval=AMP_GROWTH_INTERVAL,
)

print(
    "All labeled records:",len(all_labeled_records),
    "| refit total micro-batches:",REFIT_TOTAL_BATCHES,
    "| loader batches/epoch:",len(all_loader),
)


Final KD refit target: epoch_frac=1.250 | selected_batch=2104 | selected_update=1316
Final-refit teacher construction:
 TRAIN: heldout+refit probability ensemble
 DEV  : refit teacher only (heldout DEV logits not attached)
 shape: (18987, 13)
Final refit teacher targets: (18987, 13) | mode: refit teacher only (heldout DEV logits not attached)
✅ Saved final KD teacher targets: /kaggle/working/phase3_exp6_kd_rslora_teacher_model2_student/teacher_logits_train_dev_ensemble_for_kd.npz
tokenized 18987 records in 0.4s | teacher: True
All labeled records: 18987 | refit total micro-batches: 11864 | loader batches/epoch: 9494


In [36]:
# ============================================================
# CELL 19 — Final train+dev KD refit with exact fractional replay
# ============================================================
final_optimizer.zero_grad(set_to_none=True)

refit_start=time.time()
refit_updates=0
refit_amp_overflows=0
seen_batches=0
done=False

max_refit_epochs=int(
    math.ceil(
        REFIT_TOTAL_BATCHES/len(all_loader)
    )
)+1

for epoch in range(1,max_refit_epochs+1):
    if hasattr(all_loader.batch_sampler,"set_epoch"):
        all_loader.batch_sampler.set_epoch(epoch)

    final_student.train()
    run_total=0.0
    run_hard=0.0
    run_kd=0.0
    n_batches=0

    for step,batch in enumerate(all_loader,1):
        if seen_batches>=REFIT_TOTAL_BATCHES:
            done=True
            break

        labels=batch["labels"].to(
            DEVICE,non_blocking=True
        )
        teacher_logits=batch["teacher_logits"].to(
            DEVICE,non_blocking=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=amp_enabled,
        ):
            student_logits=_forward_batch(
                final_student,batch
            )
            hard=hard_loss_fn(
                student_logits,
                labels,
                class_w_all,
            )
            soft=kd_kl_loss(
                student_logits,
                teacher_logits,
                labels,
                KD_TEMPERATURE,
                class_w_all,
            )
            loss=(
                HARD_LOSS_WEIGHT*hard
                +KD_LOSS_WEIGHT*soft
            )/GRAD_ACCUM_STEPS

        assert bool(torch.isfinite(
            loss.detach()
        ).item())

        final_scaler.scale(loss).backward()

        run_total+=float(
            loss.detach().float().cpu()
        )*GRAD_ACCUM_STEPS
        run_hard+=float(
            hard.detach().float().cpu()
        )
        run_kd+=float(
            soft.detach().float().cpu()
        )
        n_batches+=1
        seen_batches+=1

        do_update=(
            seen_batches%GRAD_ACCUM_STEPS==0
            or seen_batches>=REFIT_TOTAL_BATCHES
        )
        if do_update:
            final_scaler.unscale_(final_optimizer)
            gn=torch.nn.utils.clip_grad_norm_(
                [
                    p for p in final_student.parameters()
                    if p.requires_grad
                ],
                GRAD_CLIP,
            )
            grad_is_finite=bool(
                torch.isfinite(torch.as_tensor(gn)).item()
            )

            old_scale=float(final_scaler.get_scale())
            final_scaler.step(final_optimizer)
            final_scaler.update()
            new_scale=float(final_scaler.get_scale())
            final_optimizer.zero_grad(set_to_none=True)

            update_succeeded=(new_scale>=old_scale) and grad_is_finite
            if update_succeeded:
                final_scheduler.step()
                refit_updates+=1
            else:
                refit_amp_overflows+=1
                print(
                    f"⚠️ Refit AMP overflow handled at batch={seen_batches} "
                    f"| grad_norm={float(torch.as_tensor(gn))} "
                    f"| scale {old_scale:.0f} -> {new_scale:.0f} "
                    f"| optimizer update skipped "
                    f"| total_refit_overflows={refit_amp_overflows}",
                    flush=True,
                )

        if (
            seen_batches%PRINT_EVERY==0
            or seen_batches>=REFIT_TOTAL_BATCHES
        ):
            elapsed=time.time()-refit_start
            rate=elapsed/max(seen_batches,1)
            eta=rate*(
                REFIT_TOTAL_BATCHES-seen_batches
            )
            print(
                f"refit batch={seen_batches}/{REFIT_TOTAL_BATCHES} "
                f"epoch≈{seen_batches/len(all_loader):.3f} "
                f"loss={run_total/max(n_batches,1):.4f} "
                f"hard={run_hard/max(n_batches,1):.4f} "
                f"kd={run_kd/max(n_batches,1):.4f} "
                f"elapsed={fmt_sec(elapsed)} "
                f"ETA≈{fmt_sec(eta)}",
                flush=True,
            )

        if seen_batches>=REFIT_TOTAL_BATCHES:
            done=True
            break

    if done:
        break

assert seen_batches==REFIT_TOTAL_BATCHES,(
    seen_batches,REFIT_TOTAL_BATCHES
)

torch.save({
    "student_state":student_state_dict(
        final_student
    ),
    "student_phase1_checkpoint":str(PHASE1_CKPT),
    "teacher_logits_path":str(ROOT/"teacher_logits_train_dev_ensemble_for_kd.npz"),
    "teacher_mode":FINAL_REFIT_TEACHER_MODE,
    "source_best_epoch_frac":BEST_FRAC,
    "source_best_batch_in_epoch":BEST_BATCH,
    "source_best_global_update":BEST_UPDATE,
    "refit_total_batches":REFIT_TOTAL_BATCHES,
    "refit_optimizer_updates":refit_updates,
    "refit_amp_overflows":int(refit_amp_overflows),
    "final_amp_scale":float(final_scaler.get_scale()),
    "source_best_dev_f1":best_f1,
    "kd_temperature":KD_TEMPERATURE,
    "hard_loss_weight":HARD_LOSS_WEIGHT,
    "kd_loss_weight":KD_LOSS_WEIGHT,
    "labels":LABELS,
    "label_to_id":LABEL_TO_ID,
    "experiment":"Phase3 Exp6 KD final train+dev refit",
},FINAL_CKPT)

print(
    "✅ Final KD refit saved:",
    FINAL_CKPT,
    "| elapsed:",
    fmt_sec(time.time()-refit_start),
    "| refit_overflows:",
    refit_amp_overflows,
    "| final_amp_scale:",
    final_scaler.get_scale()
)


refit batch=250/11864 epoch≈0.026 loss=2.8475 hard=2.4157 kd=3.1354 elapsed=00:00:15 ETA≈00:11:58
refit batch=500/11864 epoch≈0.053 loss=2.5947 hard=2.2281 kd=2.8391 elapsed=00:00:30 ETA≈00:11:41
refit batch=750/11864 epoch≈0.079 loss=2.4446 hard=2.1100 kd=2.6676 elapsed=00:00:46 ETA≈00:11:21
refit batch=1000/11864 epoch≈0.105 loss=2.3389 hard=2.0231 kd=2.5494 elapsed=00:01:01 ETA≈00:11:04
refit batch=1250/11864 epoch≈0.132 loss=2.2300 hard=1.9401 kd=2.4233 elapsed=00:01:16 ETA≈00:10:47
refit batch=1500/11864 epoch≈0.158 loss=2.1420 hard=1.8828 kd=2.3147 elapsed=00:01:31 ETA≈00:10:30
refit batch=1750/11864 epoch≈0.184 loss=2.0765 hard=1.8332 kd=2.2386 elapsed=00:01:46 ETA≈00:10:15
refit batch=2000/11864 epoch≈0.211 loss=2.0342 hard=1.8066 kd=2.1859 elapsed=00:02:01 ETA≈00:09:59
refit batch=2250/11864 epoch≈0.237 loss=1.9843 hard=1.7734 kd=2.1248 elapsed=00:02:16 ETA≈00:09:43
refit batch=2500/11864 epoch≈0.263 loss=1.9434 hard=1.7517 kd=2.0712 elapsed=00:02:31 ETA≈00:09:28
refit batch=2

In [37]:
# ============================================================
# CELL 20 — Build test sentence records
# ============================================================

def make_test_records(test_dict, domain):
    records = []
    for raw_id, sents in test_dict.items():
        did = normalize_id(raw_id)
        assert isinstance(sents, list)
        for i, sent in enumerate(sents):
            records.append({
                "doc_id": did,
                "sent_idx": i,
                "domain": domain,
                "prev": " ".join(sents[max(0, i-CONTEXT_SENTS):i]) if CONTEXT_SENTS > 0 else "",
                "target": sent,
                "next": " ".join(sents[i+1:i+1+CONTEXT_SENTS]) if CONTEXT_SENTS > 0 else "",
                "pos": position_features(i, len(sents)),
            })
    return records

test_records = (
    make_test_records(cl_test_raw, "CL")
    + make_test_records(it_test_raw, "IT")
)

test_loader = make_loader(test_records, False, EVAL_BATCH_SIZE, False)

print("Total test sentence records:", len(test_records))
assert len(test_records) == (
    sum(len(v) for v in cl_test_raw.values())
    + sum(len(v) for v in it_test_raw.values())
)


tokenized 2197 records in 0.4s | teacher: False
Total test sentence records: 2197


In [38]:
# ============================================================
# CELL 21 — KD student test inference and ensemble variants
# ============================================================
domain_stats_all=build_domain_stats(
    cl_train_raw+cl_dev_raw,
    it_train_raw+it_dev_raw,
)

def _logp(lg):
    return log_softmax_np(
        lg.astype(np.float64)
    )

def decode_to_prediction(lg,meta,name):
    ids=decode_docs(
        lg,meta,domain_stats_all,
        DECODE_ALPHA,DECODE_BETA
    )
    pred=records_to_prediction(meta,ids)
    cnt=Counter(ids.tolist())
    print(
        f"[{name}] predicted label counts:",
        {LABELS[k]:cnt[k] for k in sorted(cnt)}
    )
    return pred

lg_held,meta_held,_=predict_logits(
    student,test_loader
)
lg_refit,meta_refit,_=predict_logits(
    final_student,test_loader
)
assert meta_held==meta_refit

np.savez_compressed(
    ROOT/"test_logits.npz",
    heldout=lg_held,
    refit=lg_refit,
    doc_id=np.asarray(
        [x[0] for x in meta_held],
        dtype=object,
    ),
    sent_idx=np.asarray(
        [x[1] for x in meta_held],
        dtype=np.int64,
    ),
    domain=np.asarray(
        [x[2] for x in meta_held],
        dtype=object,
    ),
)

predictions={
    "heldout":decode_to_prediction(
        lg_held,meta_held,"heldout"
    ),
    "refit":decode_to_prediction(
        lg_refit,meta_refit,"refit"
    ),
}

if ENSEMBLE_HELDOUT_AND_REFIT:
    ensemble_logp=0.5*(
        _logp(lg_held)+_logp(lg_refit)
    )
    predictions["ensemble"]=decode_to_prediction(
        ensemble_logp,
        meta_held,
        "ensemble",
    )

# Top-k KD snapshot ensemble.
if MAKE_SNAPSHOT_ENSEMBLES:
    meta_path=ROOT/"top_snapshots.json"
    if meta_path.is_file():
        top_meta=json.loads(
            meta_path.read_text(encoding="utf-8")
        )
        snap_logits=[]

        for info in top_meta[:TOPK_HELDOUT_SNAPSHOTS]:
            sp=Path(info["path"])
            if not sp.is_file():
                continue

            snap_student=build_fresh_student()
            sobj=torch.load(
                sp,map_location="cpu",weights_only=False
            )
            snap_student.load_state_dict(
                sobj["student_state"],
                strict=True,
            )

            lg,meta_snap,_=predict_logits(
                snap_student,test_loader
            )
            assert meta_snap==meta_held
            snap_logits.append(lg)

            del snap_student
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        if len(snap_logits)>=2:
            snapshot_logp=np.mean(
                np.stack([
                    _logp(x) for x in snap_logits
                ],axis=0),
                axis=0,
            )
            predictions[
                "snapshot_ensemble"
            ]=decode_to_prediction(
                snapshot_logp,
                meta_held,
                "snapshot_ensemble",
            )

            predictions[
                "snapshot_refit_ensemble"
            ]=decode_to_prediction(
                0.5*(
                    snapshot_logp
                    +_logp(lg_refit)
                ),
                meta_held,
                "snapshot_refit_ensemble",
            )

            np.savez_compressed(
                ROOT/"snapshot_test_logits.npz",
                logp=snapshot_logp.astype(np.float32),
                n_snapshots=np.asarray(
                    [len(snap_logits)],
                    dtype=np.int64,
                ),
            )

prediction=predictions["refit"]
with open(OFFICIAL_PRED,"w",encoding="utf-8") as f:
    json.dump(
        prediction,f,
        ensure_ascii=False,indent=2
    )

print(
    "KD submission variants:",
    list(predictions)
)


[heldout] predicted label counts: {'Fact': 492, 'Issue': 14, 'ArgumentPetitioner': 254, 'ArgumentRespondent': 30, 'Statute': 139, 'Dissent': 19, 'PrecedentReliedUpon': 342, 'PrecedentNotReliedUpon': 22, 'PrecedentOverruled': 16, 'RulingByLowerCourt': 122, 'RatioOfTheDecision': 684, 'RulingByPresentCourt': 39, 'None': 24}
[refit] predicted label counts: {'Fact': 429, 'Issue': 12, 'ArgumentPetitioner': 171, 'ArgumentRespondent': 104, 'Statute': 154, 'Dissent': 8, 'PrecedentReliedUpon': 211, 'PrecedentNotReliedUpon': 33, 'PrecedentOverruled': 7, 'RulingByLowerCourt': 266, 'RatioOfTheDecision': 739, 'RulingByPresentCourt': 28, 'None': 35}
[ensemble] predicted label counts: {'Fact': 455, 'Issue': 16, 'ArgumentPetitioner': 203, 'ArgumentRespondent': 68, 'Statute': 149, 'Dissent': 20, 'PrecedentReliedUpon': 291, 'PrecedentNotReliedUpon': 25, 'PrecedentOverruled': 18, 'RulingByLowerCourt': 176, 'RatioOfTheDecision': 721, 'RulingByPresentCourt': 28, 'None': 27}
[snapshot_ensemble] predicted lab

In [39]:
# ============================================================
# CELL 21b — OPTIONAL leaderboard-only KD student + rsLoRA teacher blends
# ============================================================
# IMPORTANT:
# - These are bonus leaderboard candidates, NOT the pure KD experiment result.
# - The rsLoRA teacher's best standard ensemble was created in its own notebook as:
#       teacher_ensemble_logits = 0.5 * heldout_logits + 0.5 * refit_logits
#   so we reproduce that exact ensemble definition here.
# - Only an explicitly identifiable rsLoRA/teacher test_logits.npz is accepted.
# - Alignment is by (normalized_doc_id, sent_idx, domain), never just doc/sentence.

def _normalized_meta_key(doc_id, sent_idx, domain):
    return (
        normalize_id(str(doc_id)),
        int(sent_idx),
        str(domain),
    )

def _validate_external_teacher_test_npz(path, target_meta):
    z=np.load(path,allow_pickle=True)

    required={"heldout","refit","doc_id","sent_idx","domain"}
    missing=required-set(z.files)
    assert not missing,f"{path}: missing arrays {sorted(missing)}"

    held=np.asarray(z["heldout"],dtype=np.float64)
    refit=np.asarray(z["refit"],dtype=np.float64)

    assert held.shape==refit.shape,(
        path,held.shape,refit.shape
    )
    assert held.ndim==2 and held.shape[1]==NUM_LABELS,(
        path,held.shape,NUM_LABELS
    )
    assert np.isfinite(held).all(),f"{path}: heldout logits contain NaN/Inf"
    assert np.isfinite(refit).all(),f"{path}: refit logits contain NaN/Inf"

    src_keys=[
        _normalized_meta_key(d,s,dom)
        for d,s,dom in zip(
            z["doc_id"],z["sent_idx"],z["domain"]
        )
    ]
    tgt_keys=[
        _normalized_meta_key(d,s,dom)
        for d,s,dom in target_meta
    ]

    assert len(src_keys)==held.shape[0],(
        path,len(src_keys),held.shape[0]
    )
    assert len(set(src_keys))==len(src_keys),(
        f"{path}: duplicate external teacher rows"
    )
    assert len(set(tgt_keys))==len(tgt_keys),(
        "Current test metadata contains duplicate rows"
    )

    src_map={k:i for i,k in enumerate(src_keys)}
    src_set=set(src_keys)
    tgt_set=set(tgt_keys)

    assert src_set==tgt_set,(
        f"{path}: teacher/current test row sets differ | "
        f"missing={len(tgt_set-src_set)} extra={len(src_set-tgt_set)}"
    )

    order=np.asarray(
        [src_map[k] for k in tgt_keys],
        dtype=np.int64,
    )
    assert len(order)==len(target_meta)

    return held[order],refit[order]


# Find only external test_logits.npz files that look like the rsLoRA teacher run.
teacher_candidates=[
    p for p in find_files(
        lambda fn,p: (
            fn.lower()=="test_logits.npz"
            and str(ROOT) not in p
            and any(
                w in p.lower()
                for w in ["rslora","model4","teacher"]
            )
        )
    )
]

print("External rsLoRA teacher test-logit candidates:")
for p in teacher_candidates:
    print(" ",p)

# Prefer an rsLoRA path strongly. If several remain, refuse to guess.
ranked=sorted(
    teacher_candidates,
    key=lambda p:(
        sum(
            w in p.lower()
            for w in [
                "rslora",
                "phase3_model4_rslora_teacher",
                "model4",
                "teacher",
            ]
        ),
        p,
    ),
    reverse=True,
)

if not ranked:
    print(
        "No external rsLoRA teacher test_logits.npz attached; "
        "bonus teacher blends skipped."
    )
else:
    top_score=sum(
        w in ranked[0].lower()
        for w in [
            "rslora",
            "phase3_model4_rslora_teacher",
            "model4",
            "teacher",
        ]
    )
    equally_ranked=[
        p for p in ranked
        if sum(
            w in p.lower()
            for w in [
                "rslora",
                "phase3_model4_rslora_teacher",
                "model4",
                "teacher",
            ]
        )==top_score
    ]

    if len(equally_ranked)>1:
        print(
            "⚠️ Multiple equally plausible rsLoRA teacher test_logits.npz files found."
        )
        print(
            "Bonus blend skipped rather than silently choosing the wrong run."
        )
        for p in equally_ranked:
            print(" ",p)
    else:
        teacher_path=ranked[0]
        print("Using external rsLoRA teacher logits:",teacher_path)

        t_held,t_refit=_validate_external_teacher_test_npz(
            teacher_path,
            meta_held,
        )

        # Faithfully reproduce the winning rsLoRA standard ensemble.
        teacher_ensemble_logits=0.5*t_held+0.5*t_refit

        # Pure KD student's standard heldout+refit ensemble, also in raw-logit space.
        student_ensemble_logits=0.5*(
            lg_held.astype(np.float64)
            +lg_refit.astype(np.float64)
        )

        assert (
            teacher_ensemble_logits.shape
            ==student_ensemble_logits.shape
            ==(len(meta_held),NUM_LABELS)
        )

        # Bonus blends. These do not alter any pure-KD candidate.
        for sw in BONUS_TEACHER_BLEND_STUDENT_WEIGHTS:
            sw=float(sw)
            assert 0.0<sw<1.0
            tw=1.0-sw

            blended=(
                sw*student_ensemble_logits
                +tw*teacher_ensemble_logits
            )

            name=(
                "kd_teacher_blend_"
                f"s{int(round(100*sw)):02d}_"
                f"t{int(round(100*tw)):02d}"
            )
            predictions[name]=decode_to_prediction(
                blended,
                meta_held,
                name,
            )

        np.savez_compressed(
            ROOT/"bonus_kd_teacher_blend_logits.npz",
            student_ensemble=student_ensemble_logits.astype(np.float32),
            teacher_ensemble=teacher_ensemble_logits.astype(np.float32),
            doc_id=np.asarray(
                [x[0] for x in meta_held],
                dtype=object,
            ),
            sent_idx=np.asarray(
                [x[1] for x in meta_held],
                dtype=np.int64,
            ),
            domain=np.asarray(
                [x[2] for x in meta_held],
                dtype=object,
            ),
            student_weights=np.asarray(
                BONUS_TEACHER_BLEND_STUDENT_WEIGHTS,
                dtype=np.float32,
            ),
        )

        print(
            "✅ Bonus teacher-blend candidates added:",
            [
                k for k in predictions
                if k.startswith("kd_teacher_blend_")
            ]
        )


External rsLoRA teacher test-logit candidates:
  /kaggle/input/datasets/aaryant23/phase3-kd-model4-2/test_logits.npz
Using external rsLoRA teacher logits: /kaggle/input/datasets/aaryant23/phase3-kd-model4-2/test_logits.npz
[kd_teacher_blend_s30_t70] predicted label counts: {'Fact': 435, 'Issue': 17, 'ArgumentPetitioner': 254, 'ArgumentRespondent': 44, 'Statute': 184, 'Dissent': 12, 'PrecedentReliedUpon': 324, 'PrecedentNotReliedUpon': 19, 'PrecedentOverruled': 7, 'RulingByLowerCourt': 162, 'RatioOfTheDecision': 678, 'RulingByPresentCourt': 39, 'None': 22}
[kd_teacher_blend_s40_t60] predicted label counts: {'Fact': 438, 'Issue': 15, 'ArgumentPetitioner': 256, 'ArgumentRespondent': 41, 'Statute': 181, 'Dissent': 12, 'PrecedentReliedUpon': 323, 'PrecedentNotReliedUpon': 19, 'PrecedentOverruled': 12, 'RulingByLowerCourt': 159, 'RatioOfTheDecision': 681, 'RulingByPresentCourt': 39, 'None': 21}
[kd_teacher_blend_s50_t50] predicted label counts: {'Fact': 448, 'Issue': 17, 'ArgumentPetitioner'

In [40]:
# ============================================================
# CELL 22 — Strict submission validation + ZIP
# Mirrors evaluator structural rules for private test data.
# ============================================================

def normalized_test_map(raw):
    out = {}
    for raw_id, sents in raw.items():
        did = normalize_id(raw_id)
        assert did not in out, f"Duplicate normalized ID: {did}"
        out[did] = sents
    return out

cl_test = normalized_test_map(cl_test_raw)
it_test = normalized_test_map(it_test_raw)

assert not (set(cl_test) & set(it_test)), "CL/IT test IDs overlap after normalization."

expected = {}
expected.update(cl_test)
expected.update(it_test)

assert set(prediction) == set(expected), (
    f"ID mismatch: missing={set(expected)-set(prediction)}, "
    f"extra={set(prediction)-set(expected)}"
)

allowed = set(LABELS)
for doc_id, sents in expected.items():
    pred = prediction[doc_id]
    assert isinstance(pred, list), doc_id
    assert len(pred) == len(sents), (
        doc_id, len(pred), len(sents)
    )
    assert all(x in allowed for x in pred), doc_id

with zipfile.ZipFile(
    SUBMISSION_ZIP,
    "w",
    zipfile.ZIP_DEFLATED
) as z:
    z.write(OFFICIAL_PRED, arcname="prediction.json")

with zipfile.ZipFile(SUBMISSION_ZIP, "r") as z:
    names = z.namelist()
    assert names == ["prediction.json"], names
    zipped = json.loads(z.read("prediction.json").decode("utf-8"))

assert zipped == prediction

print("✅ FINAL SUBMISSION PASSED STRUCTURAL CHECKS")
print("Documents:", len(prediction))
print("Sentences:", sum(len(v) for v in prediction.values()))
print("ZIP contents:", names)
print("prediction.json:", OFFICIAL_PRED)
print("submission.zip :", SUBMISSION_ZIP)

# ---- also write the two variants separately (upload both; keep the better)
def write_variant(name, pred):
    assert set(pred) == set(expected)
    for doc_id, sents in expected.items():
        assert len(pred[doc_id]) == len(sents), (name, doc_id)
        assert all(x in allowed for x in pred[doc_id]), (name, doc_id)
    jp = ROOT / f"prediction_{name}.json"
    zp = ROOT / f"submission_{name}.zip"
    with open(jp, "w", encoding="utf-8") as f:
        json.dump(pred, f, ensure_ascii=False)
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jp, arcname="prediction.json")
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"]
    print("✅", zp.name)

for name, pred in predictions.items():
    write_variant(name, pred)
print("Variants written:", ", ".join(f"submission_{k}.zip" for k in predictions), "| submission.zip == refit")

# Sanity: how much do the variants disagree at sentence level?
def _flat(p): return [x for d in sorted(expected) for x in p[d]]
_names = list(predictions)
for i in range(len(_names)):
    for j in range(i + 1, len(_names)):
        a, b = _flat(predictions[_names[i]]), _flat(predictions[_names[j]])
        print(f"agreement {_names[i]} vs {_names[j]}: {np.mean([x == y for x, y in zip(a, b)]):.4f}")


# Evaluator-style structural contract check on TEST.
# Test gold labels are private, so only IDs, lengths and allowed labels can be verified.
def test_submission_contract(prediction_obj):
    if not isinstance(prediction_obj, dict):
        raise ValueError(
            "prediction.json must be a JSON object mapping document IDs to label lists."
        )

    normalized = {}

    for raw_id, labels in prediction_obj.items():
        did = normalize_id(raw_id)

        if did in normalized:
            raise ValueError(
                f"Duplicate document ID after .txt normalization: {did}"
            )

        normalized[did] = labels

    if set(normalized) != set(expected):
        missing = sorted(set(expected) - set(normalized))
        extra = sorted(set(normalized) - set(expected))
        raise ValueError(
            f"ID mismatch: missing={len(missing)}, extra={len(extra)}"
        )

    for did, source_sentences in expected.items():
        pred = normalized[did]

        if not isinstance(pred, list):
            raise ValueError(f"{did}: labels must be a JSON list.")

        if len(pred) != len(source_sentences):
            raise ValueError(
                f"{did}: sentence-count mismatch "
                f"({len(pred)} vs {len(source_sentences)})."
            )

        if any(x not in allowed for x in pred):
            raise ValueError(f"{did}: unknown label encountered.")

    return True

assert test_submission_contract(prediction)
print(
    "✅ TEST prediction satisfies the same ID / sentence-count / "
    "allowed-label contract enforced by the Phase-3 evaluator."
)



# Compact reproducibility manifest.
manifest={
    "experiment":"Phase3_Exp6_KD_rsLoRA_EnsembleTeacher_Model2_Student_v4",
    "technique":"Knowledge Distillation",
    "teacher":"Model4 + rsLoRA ensemble-aware KD",
    "student":"Phase1 Model2 decoder-only",
    "teacher_train_logits":str(TEACHER_TRAIN_PATH),
    "teacher_refit_logits":str(TEACHER_REFIT_PATH),
    "teacher_dev_heldout_logits":str(TEACHER_DEV_HELDOUT_PATH) if TEACHER_DEV_HELDOUT_PATH else None,
    "teacher_fusion_space":TEACHER_FUSION_SPACE,
    "teacher_ensemble_weights":[TEACHER_ENSEMBLE_WEIGHT_HELDOUT,TEACHER_ENSEMBLE_WEIGHT_REFIT],
    "final_refit_teacher_mode":FINAL_REFIT_TEACHER_MODE,
    "temperature":KD_TEMPERATURE,
    "hard_loss_weight":HARD_LOSS_WEIGHT,
    "kd_loss_weight":KD_LOSS_WEIGHT,
    "kd_weight_by_class":KD_WEIGHT_BY_CLASS,
    "student_backbone_lr":STUDENT_BACKBONE_LR,
    "student_layerwise_lr_decay":STUDENT_LAYERWISE_LR_DECAY,
    "student_head_lr":STUDENT_HEAD_LR,
    "head_type":"packed BiGRU",
    "head_hidden":HEAD_HIDDEN,
    "label_smoothing":LABEL_SMOOTHING,
    "best_epoch":int(best_obj["best_epoch"]),
    "best_epoch_frac":float(
        best_obj.get("best_frac",best_obj["best_epoch"])
    ),
    "best_batch_in_epoch":int(
        best_obj.get("best_batch_in_epoch",0)
    ),
    "best_global_update":int(
        best_obj.get("best_global_update",0)
    ),
    "best_dev_macro_f1":float(
        best_obj["best_macro_f1"]
    ),
    "decode_alpha":float(DECODE_ALPHA),
    "decode_beta":float(DECODE_BETA),
    "context_sents_each_side":CONTEXT_SENTS,
    "max_len":MAX_LEN,
    "effective_batch_size":MICRO_BATCH_SIZE*GRAD_ACCUM_STEPS,
    "amp_init_scale":AMP_INIT_SCALE,
    "amp_growth_interval":AMP_GROWTH_INTERVAL,
    "main_amp_overflows":int(amp_overflows),
    "refit_amp_overflows":int(refit_amp_overflows),
    "bonus_teacher_blend_student_weights":BONUS_TEACHER_BLEND_STUDENT_WEIGHTS,
    "bonus_teacher_blend_is_official_kd_result":False,
    "seed":SEED,
    "tokenizer_sha256":sha256_file(TOKENIZER_FILE),
    "student_phase1_checkpoint":str(PHASE1_CKPT),
    "artifacts":sorted([
        p.name for p in ROOT.iterdir()
        if p.is_file()
    ]),
}
with open(
    ROOT/"experiment_manifest.json",
    "w",encoding="utf-8"
) as f:
    json.dump(manifest,f,indent=2)

print(
    "✅ Saved KD experiment manifest:",
    ROOT/"experiment_manifest.json"
)


✅ FINAL SUBMISSION PASSED STRUCTURAL CHECKS
Documents: 10
Sentences: 2197
ZIP contents: ['prediction.json']
prediction.json: /kaggle/working/phase3_exp6_kd_rslora_teacher_model2_student/prediction.json
submission.zip : /kaggle/working/phase3_exp6_kd_rslora_teacher_model2_student/submission.zip
✅ submission_heldout.zip
✅ submission_refit.zip
✅ submission_ensemble.zip
✅ submission_snapshot_ensemble.zip
✅ submission_snapshot_refit_ensemble.zip
✅ submission_kd_teacher_blend_s30_t70.zip
✅ submission_kd_teacher_blend_s40_t60.zip
✅ submission_kd_teacher_blend_s50_t50.zip
Variants written: submission_heldout.zip, submission_refit.zip, submission_ensemble.zip, submission_snapshot_ensemble.zip, submission_snapshot_refit_ensemble.zip, submission_kd_teacher_blend_s30_t70.zip, submission_kd_teacher_blend_s40_t60.zip, submission_kd_teacher_blend_s50_t50.zip | submission.zip == refit
agreement heldout vs refit: 0.7346
agreement heldout vs ensemble: 0.8721
agreement heldout vs snapshot_ensemble: 0.888

## What to keep after Exp6 v5 STABLE KD

Download:
- completed KD notebook
- `best_kd_student.pt`
- `final_refit_kd_student.pt`
- `history.json`
- `top_snapshots.json`
- all `kd_snapshot_*.pt`
- `dev_logits.npz`
- `test_logits.npz`
- `snapshot_test_logits.npz` if created
- `teacher_logits_train_dev_ensemble_for_kd.npz`
- `experiment_manifest.json`
- all generated pure-KD submission ZIP variants
- `bonus_kd_teacher_blend_logits.npz` and bonus blend ZIPs only if Cell 21b found the correct rsLoRA teacher file

### Teacher logic

**Heldout KD model selection**
- TRAIN only
- teacher = `teacher_logits_train_heldout.npz`
- the refit teacher is deliberately excluded
- DEV therefore remains a genuine holdout

**Final train+dev KD refit**
- TRAIN examples: 50/50 probability ensemble of heldout + refit rsLoRA teachers
- DEV examples:
  - heldout+refit ensemble if `teacher_logits_dev_heldout.npz` is attached
  - otherwise refit teacher only
- exact final targets are saved to `teacher_logits_train_dev_ensemble_for_kd.npz`

### Official Exp6 KD Codabench candidates

These are the **pure Model2 KD student** outputs and are the ones to use when reporting Experiment 6:
- `submission_heldout.zip`
- `submission_refit.zip`
- `submission_ensemble.zip`
- `submission_snapshot_ensemble.zip`
- `submission_snapshot_refit_ensemble.zip`

### Bonus leaderboard-only candidates

If the original rsLoRA `test_logits.npz` is attached, v4 may additionally create:
- `submission_kd_teacher_blend_s30_t70.zip`
- `submission_kd_teacher_blend_s40_t60.zip`
- `submission_kd_teacher_blend_s50_t50.zip`

These reuse Model4 teacher predictions at inference time, so they are **not** the pure KD result. They are only optional leaderboard candidates.

The bonus blend now reproduces the rsLoRA teacher's actual standard ensemble definition:
`0.5 * heldout_logits + 0.5 * refit_logits`,
with strict full-key row alignment.


### FP16 stability behavior

On a Tesla T4, an occasional gradient overflow can happen during full Model2 fine-tuning.
v5 does **not** terminate the run for an isolated AMP overflow. `GradScaler` skips that optimizer
update, reduces its scale, clears the bad gradients, and training continues.

The notebook uses:
- microbatch `2`
- grad accumulation `8`
- effective batch `16`
- AMP initial scale `4096`

Watch the printed `AMP overflow handled...` messages. A few are acceptable; persistent/repeated
overflows every few updates would indicate a deeper numerical issue.
